# 深圳市南山区 15 分钟城市时间贫困研究

**— 基于改进两步移动搜索法 (Modified 2SFCA) 与路网加权可达性的时空公平性分析 —**

**Target Journal**: *Computers, Environment and Urban Systems* (CEUS, JCR Q1)
**研究区域**: 深圳市南山区  
**数据来源**: OSMnx 路网、OSM POI、大众点评 POI  
**核心方法**: Modified 2SFCA · Gaussian 2SFCA · Hansen Integral Accessibility · 路网 Dijkstra 加权  
**统计检验**: Moran's I · LISA Cluster · ANOVA / Kruskal-Wallis · Bootstrap CI

---

**研究特色 — Science 与 Humanistic Care 并重**:

| 维度 | 科学维度 | 人文维度 |
|------|----------|----------|
| **核心问题** | 15分钟生活圈的政策承诺与GIS可达性测算 | 可达性"幻觉"：统计数据良好 vs 弱势群体真实体验 |
| **研究对象** | 设施覆盖率、路网效率 | 城中村居民、老年人、残障者、儿童的生活经验 |
| **方法创新** | 多模型对比 (2SFCA/Gaussian/Hansen) | 分群体加权可达性 + 脆弱性叠加分析 |
| **公平性测度** | Gini系数、LISA聚类 | 可达性剥夺指数、多维脆弱性评分 |

**研究目标**：揭示15分钟生活圈在统计可达性背后，对弱势群体（城中村居民、老年人、残障人士、儿童）的真实可达性鸿沟，并提出有温度的空间政策建议。

---

**目录**:
1. [环境配置与依赖安装](#1)
2. [OSMnx 真实路网数据获取](#2)
3. [POI 设施数据获取与预处理](#3)
4. [弱势群体画像与社会脆弱性分析](#3b)
5. [两步移动搜索可达性模型 (2SFCA)](#4)
6. [高斯衰减改进模型 (Gaussian 2SFCA)](#5)
7. [多时段可达性对比分析 (白天/夜间)](#6)
8. [公平性分析 — Gini系数与剥夺指数](#6b)
9. [空间自相关检验 (Moran's I & LISA)](#7)
10. [交互可视化 (Folium)](#8)
11. [建筑AOI分析 — Fig11 补充图表 (独立4张)](#10)
12. [政策启示与人文反思](#9)

<a id='1'></a>
---

## 1. 环境配置与依赖安装

In [ ]:
# pyrightconfig.json 位于上级目录，basedpyright 会自动识别
# 以下注释用于在 VS Code / Cursor 中临时抑制 type-stub 误报
# basedpyright: reportMissingImports=false
# basedpyright: reportMissingTypeStubs=false
# basedpyright: reportUnknownMemberType=false
# basedpyright: reportUnknownVariableType=false
# basedpyright: reportUnknownParameterType=false
# basedpyright: reportAttributeAccessIssue=false
# basedpyright: reportCallIssue=false
# basedpyright: reportGeneralTypeIssues=false
# basedpyright: reportPossiblyUnboundVariable=false


In [ ]:
# 安装所有必需依赖
# (libraries pre-installed)  # !pip install (pre-installed) -q osmnx networkx geopandas shapely pyproj folium matplotlib pandas numpy requests libpysal esda spopt saliency scipy scikit-learn

In [ ]:
# BBOX (Nanshan District boundary, WGS84) + cache path
BBOX = {'north': 22.80, 'south': 22.40, 'east': 114.45, 'west': 113.85}
cache_path = os.path.join(BASE_DIR, 'osm_cache')
os.makedirs(cache_path, exist_ok=True)
print(f'BBOX: N={BBOX["north"]}, S={BBOX["south"]}, E={BBOX["east"]}, W={BBOX["west"]}')
print(f'Cache: {cache_path}')


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
    'SimSun', 'AR PL UMing CN', 'WenQuanYi Micro Hei', 'Arial Unicode MS', 'DejaVu Sans'
]
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 100

import pandas as pd
import numpy as np
import geopandas as gpd
import os
import sys
import time
import json
import math
import folium
from folium.plugins import HeatMap, HeatMapWithTime
from folium import plugins

import scipy.stats as stats
from scipy.spatial import cKDTree
from scipy.stats import gaussian_kde
import scipy

import networkx as nx
import osmnx as ox
ox.settings.use_cache = True
ox.settings.log_console = False

from libpysal.weights import Queen
import esda
from esda.moran import Moran, Moran_Local  # type: ignore
from libpysal.weights import DistanceBand

import libpysal

BASE_DIR = r"e:\xicha gis 智能定位\15分钟城市时间贫困研究"
os.makedirs(BASE_DIR, exist_ok=True)

print("=" * 60)
print("依赖包验证")
print("=" * 60)
pkgs = {'pandas': pd, 'numpy': np, 'geopandas': gpd, 'matplotlib': matplotlib,
         'networkx': nx, 'osmnx': ox, 'scipy': scipy, 'folium': folium,
         'esda': esda, 'libpysal': libpysal}
for name, pkg in pkgs.items():
    print(f"  {name:15s}: {pkg.__version__}")
print(f"\n工作目录: {BASE_DIR}")

<a id='2'></a>
---

## 2. OSMnx 真实路网数据获取

### 2.1 研究区域定义

本节使用 OSMnx 从 OpenStreetMap 真实获取深圳市南山区的步行道路网络，
包括道路边（edges）和节点（nodes），用于基于路网距离的可达性计算，
替代直线欧氏距离，获得更真实的步行时间估计。

<a id="village_data"></a>

---

## 3b.2 真实小区数据：搜房数据集成

### 数据来源

使用搜房网 (SouFun) **2017年深圳小区数据**替代模拟数据，共 **1539个深圳小区**：

| 字段 | 来源 | 说明 |
|------|------|------|
| `housetitle` → `name` | fang_2017-08-02.sql | 小区名称 |
| `address` → `full_address` | fang_2017-08-02.sql | 详细地址 |
| `quxian` | fang_2017-08-02.sql | 区县 |
| `shangquan` → `business_district` | fang_2017-08-02.sql | 商圈 |
| `money` | fang_2017-08-02.sql | 单价（元/平米）|
| `lng/lat` | 地理编码（Nominatim/Amap） | 经纬度 |

### 小区类型推断

基于搜房单价 + 小区名称关键词组合判断：

| 类型 | 关键词 | 价格区间 |
|------|--------|----------|
| `urban_village` 城中村 | 名称含"村/城中村" | 单价 < 25,000 元/平米 |
| `affordable_housing` 保障房 | — | 25,000–45,000 元/平米 |
| `commodity_housing` 商品房 | — | 45,000–80,000 元/平米 |
| `high_end` 高端社区 | — | > 80,000 元/平米 |

### 注意事项

> ⚠️ **数据时效性**：搜房数据为2017年快照，价格和小区信息可能已有变化。
> 建议结合以下真实数据源进行校验：
> - 深圳市住建局现势楼盘数据
> - 大众点评实况设施评分
> - 深圳市统计局人口普查分区数据

### 数据质量

- **1539条** 深圳小区记录（宝安+龙岗为主）
- 经地理编码后获取经纬度
- 坐标精度： Nominatim ~10m / Amap ~5m


In [ ]:
# ============================================================================
# 【替换模拟数据】搜房真实小区数据加载
# 数据来源: fang_2017-08-02.sql (搜房网2017年深圳小区数据)
# 数据库: village_data\villages.db
# ============================================================================
#
# 小区类型推断规则:
#   urban_village (城中村): 单价 < 25000元/平米 或名称含"村/城中村"
#   affordable_housing (保障房): 单价 25000-45000元/平米
#   commodity_housing (商品房): 单价 45000-80000元/平米
#   high_end (高端社区): 单价 > 80000元/平米
#
# 字段说明:
#   housetitle -> name (小区名)
#   center_lng/center_lat (中心点坐标，与 2SFCA 兼容)
#   community_type (小区类型)
#   population (人口估算，实际需接统计局)
#   built_year (建成年份估算，实际需接住建局)
#   area_m2 (占地面积估算，实际需接规划局)
#   supply (供给指标，实际需大众点评评分)
#
# 依赖: geopandas, shapely, pandas, numpy, sqlite3
# 安装: pip install geopandas pandas numpy shapely

import os, sys, sqlite3
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

VILLAGE_DB = r"e:\\xicha gis 智能定位\\15分钟城市时间贫困研究\\village_data\\villages.db"

# 区域中心用于无坐标时估算
DISTRICT_CENTROIDS = {
    '宝安': (113.8828, 22.5553), '龙岗': (114.2471, 22.7205),
    '南山': (113.9308, 22.5332), '福田': (114.0579, 22.5435),
    '罗湖': (114.1317, 22.5482), '盐田': (114.2361, 22.5557),
    '光明': (113.9297, 22.7623), '坪山': (114.3507, 22.6802),
    '龙华': (114.0495, 22.7149), '大鹏': (114.4871, 22.5817),
}
PRICE_THRESHOLDS = [
    (0, 25000, 'urban_village'), (25000, 45000, 'affordable_housing'),
    (45000, 80000, 'commodity_housing'), (80000, float('inf'), 'high_end'),
]
URBAN_VILLAGE_KEYWORDS = ['村', '城中村', '旧村', '新村', '居民点']

def infer_community_type(name, price):
    if isinstance(name, str):
        for kw in URBAN_VILLAGE_KEYWORDS:
            if kw in name:
                return 'urban_village'
    price = float(price) if price else 0
    for lo, hi, ctype in PRICE_THRESHOLDS:
        if lo <= price < hi:
            return ctype
    return 'high_end'

def load_village_data(db_path, require_coords=True):
    if not os.path.exists(db_path):
        print(f"[ERROR] Database not found: {db_path}")
        print("  Run geocode_nominatim.py or geocode_amap.py first")
        return None
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM sz_village", conn)
    conn.close()
    print(f"SQLite records: {len(df)}")
    n_geo = df['lng'].notna().sum()
    print(f"With coordinates: {n_geo} / {len(df)}")
    if require_coords:
        df_work = df.dropna(subset=['lng', 'lat']).copy()
        df_work = df_work[
            (df_work['lng'] > 113.0) & (df_work['lng'] < 114.8) &
            (df_work['lat'] > 22.0) & (df_work['lat'] < 23.0)
        ].copy()
        print(f"Valid records: {len(df_work)}")
    else:
        def rough_coords(row):
            quxian = str(row.get('quxian', ''))
            for d, (lng, lat) in DISTRICT_CENTROIDS.items():
                if d in quxian:
                    return pd.Series([lng + np.random.uniform(-0.01, 0.01),
                                      lat + np.random.uniform(-0.01, 0.01)])
            return pd.Series([None, None])
        coords = df.apply(rough_coords, axis=1)
        df['lng'] = df['lng'].fillna(coords[0].values)
        df['lat'] = df['lat'].fillna(coords[1].values)
        df_work = df.copy()
        print("[NOTE] Using district centroids for missing coords")
    if len(df_work) == 0:
        print("[ERROR] No valid records! Check geocoding results.")
        return None
    df_work['community_type'] = df_work.apply(
        lambda r: infer_community_type(r['housetitle'], r['money']), axis=1
    )
    geometry = [Point(xy) for xy in zip(df_work['lng'], df_work['lat'])]
    gdf = gpd.GeoDataFrame(df_work, geometry=geometry, crs='EPSG:4326')
    gdf = gdf.rename(columns={
        'housetitle': 'name', 'sqpinyin': 'area_pinyin',
        'address': 'full_address', 'shangquan': 'business_district',
    })
    gdf['center_lng'] = gdf['lng']
    gdf['center_lat'] = gdf['lat']
    gdf = gdf.reset_index(drop=True)
    gdf['community_id'] = range(1, len(gdf) + 1)
    # 若数据库已有预计算字段，直接使用；否则才用推断值/随机值
    # （generate_nanshan_communities.py 已将真实感数据预计算进 villages.db）
    if "community_type" not in gdf.columns or gdf["community_type"].isna().all():
        gdf["community_type"] = gdf.apply(
            lambda r: infer_community_type(r.get("housetitle", ""), r.get("money", 0)), axis=1
        )
    if "population" not in gdf.columns or gdf["population"].isna().all():
        gdf["population"] = np.random.randint(500, 8000, size=len(gdf))
    if "built_year" not in gdf.columns or gdf["built_year"].isna().all():
        gdf["built_year"] = np.random.randint(1990, 2023, size=len(gdf))
    if "area_m2" not in gdf.columns or gdf["area_m2"].isna().all():
        gdf["area_m2"] = np.random.uniform(3000, 80000, size=len(gdf))
    if "supply" not in gdf.columns or gdf["supply"].isna().all():
        gdf["supply"] = np.random.uniform(0.5, 2.0, size=len(gdf))
    return gdf

# 加载数据（require_coords=True 表示必须先完成地理编码）
# 将 require_coords 改为 False 可使用区域中心估算进行测试
communities_gdf = load_village_data(VILLAGE_DB, require_coords=True)

if communities_gdf is not None:
    print(f"\nTotal communities: {len(communities_gdf)}")
    print("Type distribution:")
    print(communities_gdf['community_type'].value_counts())
    print(f"\n[OK] communities_gdf ready. CRS: {communities_gdf.crs}")
    print("Next: Run Cell 15+ (vulnerability profiling) then Cell 17+ (2SFCA)")
else:
    print("\n[ERROR] Failed to load village data. Check database.")


In [ ]:
# Download Nanshan walk network from OpenStreetMap
import osmnx as ox
print('Downloading Nanshan walk network from OSM...')
place_name = 'Nanshan District, Shenzhen, Guangdong, China'
graphml_path = os.path.join(cache_path, 'nanshan_walk.graphml')
if os.path.exists(graphml_path):
    print('Loading from cache...')
    G_raw = ox.load_graphml(graphml_path)
else:
    print('Downloading (first run 3-5 min)...')
    G_raw = ox.graph_from_place(place_name, network_type='walk', simplify=True, retain_all=False)
    ox.save_graphml(G_raw, graphml_path)
    print(f'Saved: {graphml_path}')
print(f'  Nodes: {len(G_raw.nodes):}')
print(f'  Edges: {len(G_raw.edges):}')


In [ ]:
# Road network preprocessing + G_walk
WALK_SPEED_M_PER_MIN = 83.33

def add_travel_time(G, speed_mpm=WALK_SPEED_M_PER_MIN):
    for u, v, data in G.edges(data=True):
        if 'length' not in data:
            data['length'] = 0
        data['travel_time_min'] = data['length'] / speed_mpm
    return G

G_walk = add_travel_time(G_raw)
nodes_gdf = ox.graph_to_gdfs(G_walk, points=False, edges=False, node_geometry=True)
edges_gdf = ox.graph_to_gdfs(G_walk, points=False, nodes=False, edge_geometry=False)
ox.save_graphml(G_walk, os.path.join(cache_path, 'nanshan_walk_network.graphml'))
print(f'Cached: {cache_path}')
print(f'Nodes: {len(G_walk.nodes):}')
print(f'Edges: {len(G_walk.edges):}')
edges_data = [d for u, v, d in G_walk.edges(data=True)]
avg_len = sum(d.get('length', 0) for d in edges_data) / max(1, len(edges_data))
total_km = sum(d.get('length', 0) for d in edges_data) / 1000
print(f'Avg edge: {avg_len:.1f}m, Total: {total_km:.1f}km')


In [ ]:
# 路网可视化
fig, ax = ox.plot_graph(
    G_walk,
    figsize=(14, 10),
    bgcolor='white',
    node_color='steelblue',
    node_size=8,
    edge_color='lightgray',
    edge_linewidth=0.5,
    show=False,
    close=False
)
ax.set_title('深圳市南山区步行道路网络 (OSMnx)', fontsize=16, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '01_nanshan_road_network.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("图表已保存: 01_nanshan_road_network.png")

<a id='3'></a>
---

## 3. POI 设施数据获取与预处理

### 3.1 从 OSM 获取真实设施 POI

使用 OSMnx 的 `geometries_from_bbox` 获取南山区内的真实公共设施 POI，
涵盖医疗、零售、教育、金融、交通等 5 大类 12 小类设施。

In [ ]:
# ============================================================================
# 从 OSM 获取真实 POI 设施
# ============================================================================

print("正在从 OpenStreetMap 获取 POI 设施数据...")

# 定义设施查询标签（OSM Tag 系统）
POI_TAGS = {
    # 医疗设施
    'hospital': {'amenity': 'hospital'},
    'clinic': {'amenity': 'clinic'},
    'pharmacy': {'amenity': 'pharmacy'},
    # 零售设施
    'supermarket': {'shop': 'supermarket'},
    'convenience': {'shop': 'convenience'},
    'market': {'amenity': 'marketplace'},
    # 教育设施
    'school': {'amenity': 'school'},
    'kindergarten': {'amenity': 'kindergarten'},
    'university': {'amenity': 'university'},
    # 金融设施
    'bank': {'amenity': 'bank'},
    'atm': {'amenity': 'atm'},
    # 交通设施
    'bus_stop': {'highway': 'bus_stop'},
    'subway': {'railway': 'station'},
}

poi_list = []
for ftype, tags in POI_TAGS.items():
    try:
        # 使用 osmnx.features_from_bbox（osmnx >= 2.0；旧版用 ox.geometries_from_bbox）
        gdf = ox.features_from_bbox(
            bbox=(BBOX['north'], BBOX['south'], BBOX['west'], BBOX['east']),
            tags=tags
        )
        # bbox 参数顺序: (north, south, west, east)
        # 旧版 osmnx 用 geometries_from_bbox 时参数顺序为 (north, south, east, west)
        if len(gdf) > 0:
            gdf = gdf.reset_index()
            gdf['facility_type'] = ftype
            gdf['name'] = gdf.get('name', f'{ftype}_{gdf.index}')
            if hasattr(gdf.geometry, 'centroid'):
                gdf['lng'] = gdf.geometry.centroid.x
                gdf['lat'] = gdf.geometry.centroid.y
            elif hasattr(gdf.geometry, 'x'):
                gdf['lng'] = gdf.geometry.x
                gdf['lat'] = gdf.geometry.y
            else:
                gdf['lng'] = gdf.geometry.apply(lambda p: p.x if hasattr(p, 'x') else None)
                gdf['lat'] = gdf.geometry.apply(lambda p: p.y if hasattr(p, 'y') else None)
            poi_list.append(gdf[['name', 'facility_type', 'lng', 'lat']].copy())
            print(f"  ✓ {ftype:15s}: {len(gdf):3d} 个")
    except Exception as e:
        print(f"  ✗ {ftype:15s}: 获取失败 ({e})")

if poi_list:
    poi_osm = pd.concat(poi_list, ignore_index=True)
    # 过滤有效坐标
    poi_osm = poi_osm.dropna(subset=['lng', 'lat'])
    poi_osm = poi_osm[
        (poi_osm['lng'] > BBOX['west']) & (poi_osm['lng'] < BBOX['east']) &
        (poi_osm['lat'] > BBOX['south']) & (poi_osm['lat'] < BBOX['north'])
    ]
    poi_osm['source'] = 'OSM'
    print(f"\nOSM POI 汇总: {len(poi_osm)} 个设施")
else:
    print("OSM POI 获取失败，使用模拟数据")
    poi_osm = None

In [ ]:
# ============================================================================
# 加载真实 POI 数据（nanshan_poi_integrated.csv）
# 数据来源：final_integrate.py（高德 API 采集 + ground truth 融合）
# 包含：设施名称、分类、坐标、火星坐标系、设施类型、夜间服务标注
# ============================================================================

POI_INTEGRATED_PATH = os.path.join(BASE_DIR, 'osm_data', 'nanshan_poi_integrated_v2.csv')



def generate_supplementary_poi(bbox, n_poi=200, seed=42):
    """
    当 nanshan_poi_integrated_v2.csv 不存在时的模拟 POI 备用生成器。
    按南山区设施密度分布生成模拟 POI，确保各类型比例合理。
    """
    np.random.seed(seed)
    POI_TYPES = {
        "便利店": 30, "药店": 15, "超市": 10,
        "银行": 12, "ATM": 15, "学校": 8,
        "幼儿园": 10, "医院": 3, "诊所": 8,
        "公交站": 40, "地铁站": 5,
        "餐饮": 25, "休闲娱乐": 10, "购物": 15
    }
    rows = []
    for ftype, count in POI_TYPES.items():
        lats = np.random.uniform(bbox["south"], bbox["north"], count)
        lngs = np.random.uniform(bbox["west"], bbox["east"], count)
        for j in range(count):
            rows.append({
                "name": f"{ftype}_{j+1}",
                "facility_type": ftype,
                "lng": lngs[j], "lat": lats[j],
                "night_service": ftype in ["便利店", "药店", "餐饮", "公交站", "地铁站"],
                "supply": round(np.random.uniform(0.8, 2.0), 2),
                "category1": "公共设施", "category2": ftype,
            })
    return pd.DataFrame(rows)


def load_real_poi(path):
    """
    加载 nanshan_poi_integrated.csv 并进行列映射
    列映射：
      gcj_lon -> lng, gcj_lat -> lat
      facility_type (已有) 保留
      night_service_final -> night_service
      supply 从 facility_type 推导（已通过 v2.csv 的精细化夜间标注优化）
    """
    df = pd.read_csv(path)
    print(f"Loaded POI records: {len(df):}个")
    
    # 列映射（GCJ-02 坐标系）
    df = df.rename(columns={'gcj_lon': 'lng', 'gcj_lat': 'lat'})
    
    # 南山区边界过滤（bbox 内）
    df = df[
        (df['lng'] > BBOX['west']) & (df['lng'] < BBOX['east']) &
        (df['lat'] > BBOX['south']) & (df['lat'] < BBOX['north'])
    ].copy()
    print(f"Within BBOX: {len(df):}个")
    
    # 夜间服务列
    # night_service_final 来自精细化推断(v2): V5优先 + 名称关键词 + 类型概率推断
    if 'night_service_final' in df.columns:
        df['night_service'] = df['night_service_final'].astype(bool)
    else:
        df['night_service'] = False
    
    # 供给水平（supply）：从设施类型推导，模拟大众点评评分归一化
    SUPPLY_MAP = {
        '医疗保健': 1.8, '药店': 1.5, 'hospital': 1.8, 'clinic': 1.4, 'pharmacy': 1.5,
        '便利店': 1.2, 'convenience': 1.2, 'supermarket': 1.6, '超市': 1.6,
        '银行': 1.3, 'bank': 1.3, 'ATM': 1.4, 'atm': 1.4,
        '学校': 1.5, 'school': 1.5, 'kindergarten': 1.4, '幼儿园': 1.4,
        '大学': 1.5, 'university': 1.5,
        '公交站': 1.8, 'bus_stop': 1.8, 'subway': 1.9,
        '交通设施': 1.7, '地铁站': 1.9, '地铁': 1.9,
        '休闲娱乐': 1.4, '餐饮服务': 1.3, '购物服务': 1.2,
        '住宿服务': 1.3, '政府机构': 1.5, '公共设施': 1.4,
        '生活服务': 1.2, '公司企业': 1.0,
        '商务写字楼': 1.1, '其他': 1.0,
    }
    def get_supply(ftype):
        base = SUPPLY_MAP.get(str(ftype), 1.0)
        return base + np.random.uniform(-0.2, 0.2)
    
    if 'supply' not in df.columns or df['supply'].isna().all():
        np.random.seed(42)
        df['supply'] = df['facility_type'].apply(get_supply).clip(0.3, 2.0)
    
    # 保留必要列
    keep_cols = ['name', 'facility_type', 'lng', 'lat', 'night_service', 'supply', 'category1', 'category2']
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()
    df['source'] = 'Gaode+GroundTruth'
    
    return df

if os.path.exists(POI_INTEGRATED_PATH):
    poi_df = load_real_poi(POI_INTEGRATED_PATH)
    print("\n[OK] 使用真实 POI 数据（nanshan_poi_integrated.csv）")
else:
    print(f"[WARN] POI 文件不存在: {POI_INTEGRATED_PATH}")
    print("回退到模拟数据...")
    poi_df = generate_supplementary_poi(BBOX)

print(f"\n最终 POI 数据集: {len(poi_df):} 个设施")
print("\n设施类型分布:")
print(poi_df['facility_type'].value_counts().to_string())
print(f"\n夜间服务设施: {poi_df['night_service'].sum():}个 ({poi_df['night_service'].mean()*100:.1f}%)")


<a id='3b'></a>

---

## 3b. 弱势群体画像与社会脆弱性分析

### 3b.1 研究背景：可达性幻觉与真实体验的鸿沟

"15分钟城市"政策以平均步行距离构建了一套美好的空间承诺。然而，这套叙事背后存在显著的**可达性幻觉 (Accessibility Illusion)**：

> **核心问题**：当研究者报告"南山区80%的小区在15分钟步行范围内可达医疗设施"时，这些统计数字掩盖了一个根本性问题——谁是被统计"平均"掉的20%？谁又是在平均中被代表的那部分人？

本研究聚焦四类最容易被空间排斥的弱势群体：

| 群体 | 空间脆弱性来源 | 可达性特殊需求 |
|------|---------------|---------------|
| **城中村居民** | 高密度、人口密集、路网破碎、设施质量低 | 公平且高质量的设施可及 |
| **老年人 (≥65岁)** | 步行速度慢、行动能力受限、昼夜出行受限 | 短距离、24h/低门槛设施 |
| **残障人士** | 无障碍设施缺乏、轮椅通道不足 | 无障碍通道、语音/触觉引导 |
| **儿童 (≤12岁)** | 无法独立出行、依赖成人陪同 | 安全、亲子类设施可及 |

**"脆弱性叠加" (Vulnerability Stacking)**：当一个人同时属于多个弱势群体时（如城中村的老人），其可达性约束呈指数级恶化，而非线性叠加。

In [ ]:
# ============================================================================
# 弱势群体画像与社会脆弱性评分系统
# ============================================================================
"""
本模块构建多维脆弱性评分 (Multi-dimensional Vulnerability Index, MVI)，
将每个小区/居民点的社会脆弱性量化为可操作的GIS指标。

参考文献:
- Flanagan, B. E., et al. (2011). A social vulnerability index for disaster management.
  Journal of Homeland Security and Emergency Management.
- Cutter, S. L., et al. (2003). Social vulnerability to environmental hazards.
  Social Science Quarterly.
"""

class VulnerablePopulationProfiler:
    """
    多维脆弱性评分器
    
    维度构成:
    1. 住房脆弱性 (Housing Vulnerability)
    2. 社会经济脆弱性 (Socioeconomic Vulnerability)
    3. 空间可达脆弱性 (Spatial Access Vulnerability)
    4. 生理脆弱性 (Physiological Vulnerability)
    
    综合脆弱性指数 MVI = w1*HV + w2*SEV + w3*SAV + w4*PV
    """
    
    def __init__(self):
        # 各维度权重（可调整，反映政策优先级）
        self.weights = {
            'housing': 0.25,        # 住房脆弱性
            'socioeconomic': 0.25,  # 社会经济脆弱性
            'spatial': 0.25,       # 空间可达脆弱性
            'physiological': 0.25    # 生理脆弱性
        }
    
    def compute_housing_vulnerability(self, community_type):
        """
        住房脆弱性 HV
        基于小区类型（城中村、保障房、商品房、高端社区）的历史脆弱性评估
        """
        housing_scores = {
            'urban_village': 0.85,     # 城中村：高密度、违建多、消防隐患
            'affordable_housing': 0.55, # 保障房：低收入聚集但设施较新
            'commodity_housing': 0.30,  # 商品房：中等脆弱性
            'high_end': 0.10            # 高端社区：低脆弱性
        }
        return housing_scores.get(community_type, 0.50)
    
    def compute_socioeconomic_vulnerability(self, community_type, population, built_year):
        """
        社会经济脆弱性 SEV
        
        指标：
        - 人口密度（高密度→高脆弱性）
        - 建成年代（老旧→高脆弱性）
        - 小区类型（反映居民社会经济地位）
        """
        # 人口密度归一化（越大越脆弱）
        pop_density_score = np.clip(population / 5000, 0, 1)
        
        # 建成年代（越老越脆弱）
        age_score = np.clip((2025 - built_year) / 40, 0, 1)
        
        # 类型权重
        type_weight = {
            'urban_village': 0.9,
            'affordable_housing': 0.7,
            'commodity_housing': 0.4,
            'high_end': 0.1
        }.get(community_type, 0.5)
        
        sev = 0.4 * type_weight + 0.3 * pop_density_score + 0.3 * age_score
        return np.clip(sev, 0, 1)
    
    def compute_spatial_vulnerability(self, dist_to_center, nearest_poi_dist):
        """
        空间可达脆弱性 SAV
        
        指标：
        - 距区中心距离（越远→脆弱性越高）
        - 最近设施距离（越大→脆弱性越高）
        """
        # 距中心距离归一化（南山区域约10km×10km，边界约7km）
        center_score = np.clip(dist_to_center / 5000, 0, 1)
        
        # 最近设施距离（步行时间替代）
        poi_score = np.clip(nearest_poi_dist / 1500, 0, 1)
        
        return 0.5 * center_score + 0.5 * poi_score
    
    def compute_physiological_vulnerability(self, community_type, population):
        """
        生理脆弱性 PV
        
        基于小区类型的年龄结构推断（模拟）：
        - 城中村：多为外来务工青壮年，但老人/儿童托管需求高
        - 保障房：低收入家庭，老人和儿童比例较高
        - 高端社区：中青年白领家庭
        """
        type_profiles = {
            'urban_village': {
                'elderly_ratio': 0.12,    # 12% 老年人（随迁老人）
                'child_ratio': 0.18,     # 18% 儿童（留守儿童）
                'disability_ratio': 0.035 # 3.5% 残障人士
            },
            'affordable_housing': {
                'elderly_ratio': 0.22,    # 保障房老人比例更高
                'child_ratio': 0.20,
                'disability_ratio': 0.042
            },
            'commodity_housing': {
                'elderly_ratio': 0.15,
                'child_ratio': 0.12,
                'disability_ratio': 0.028
            },
            'high_end': {
                'elderly_ratio': 0.08,
                'child_ratio': 0.10,
                'disability_ratio': 0.020
            }
        }
        
        profile = type_profiles.get(community_type, type_profiles['commodity_housing'])
        
        # 生理脆弱性 = 加权脆弱群体比例
        # 老年人权重1.0，残障人士权重1.2（完全依赖设施），儿童权重0.8
        pv = (1.0 * profile['elderly_ratio'] + 
              1.2 * profile['disability_ratio'] + 
              0.8 * profile['child_ratio'])
        
        return np.clip(pv / 0.5, 0, 1)  # 归一化
    
    def profile_community(self, row, dist_to_center=None, nearest_poi_dist=None):
        """
        对单个小区进行完整脆弱性画像
        
        返回: dict，包含各维度评分和综合MVI
        """
        ctype = row.get('community_type', 'commodity_housing')
        pop = row.get('population', 2000)
        year = row.get('built_year', 2010)
        
        hv = self.compute_housing_vulnerability(ctype)
        sev = self.compute_socioeconomic_vulnerability(ctype, pop, year)
        pv = self.compute_physiological_vulnerability(ctype, pop)
        
        if dist_to_center is not None and nearest_poi_dist is not None:
            sav = self.compute_spatial_vulnerability(dist_to_center, nearest_poi_dist)
        else:
            # 使用默认值（后续可用实际计算替代）
            sav = 0.50
        
        # 综合脆弱性指数 MVI
        mvi = (self.weights['housing'] * hv +
               self.weights['socioeconomic'] * sev +
               self.weights['spatial'] * sav +
               self.weights['physiological'] * pv)
        
        return {
            'HV': hv,
            'SEV': sev,
            'SAV': sav,
            'PV': pv,
            'MVI': mvi
        }
    
    def profile_all_communities(self, gdf, dist_to_center_dict=None, nearest_poi_dict=None):
        """
        对所有小区批量计算脆弱性画像
        """
        results = []
        for _, row in gdf.iterrows():
            dist_c = dist_to_center_dict.get(row['community_id']) if dist_to_center_dict else None
            poi_d = nearest_poi_dict.get(row['community_id']) if nearest_poi_dict else None
            profile = self.profile_community(row, dist_c, poi_d)
            results.append(profile)
        
        profile_df = pd.DataFrame(results)
        result_gdf = gdf.copy()
        for col in profile_df.columns:
            result_gdf[col] = profile_df[col].values
        
        return result_gdf
    
    def get_group_statistics(self, gdf, group_col='community_type'):
        """
        按小区类型输出脆弱性统计摘要
        """
        groups = {
            'urban_village': '城中村',
            'affordable_housing': '保障房',
            'commodity_housing': '商品房',
            'high_end': '高端社区'
        }
        
        summary = []
        for gtype, glabel in groups.items():
            subset = gdf[gdf[group_col] == gtype]
            if len(subset) == 0:
                continue
            summary.append({
                '类型': glabel,
                '小区数': len(subset),
                'HV_均值': subset['HV'].mean(),
                'SEV_均值': subset['SEV'].mean(),
                'SAV_均值': subset['SAV'].mean(),
                'PV_均值': subset['PV'].mean(),
                'MVI_均值': subset['MVI'].mean(),
                'MVI_中位数': subset['MVI'].median(),
                'MVI_最大值': subset['MVI'].max()
            })
        
        return pd.DataFrame(summary).sort_values('MVI_均值', ascending=False)


# 执行脆弱性评分
profiler = VulnerablePopulationProfiler()
communities_gdf = profiler.profile_all_communities(communities_gdf)

print("=" * 70)
print("弱势群体脆弱性分析 — 多维脆弱性指数 (MVI)")
print("=" * 70)

mvi_stats = profiler.get_group_statistics(communities_gdf)
print("\n按小区类型的脆弱性均值排序:")
print(mvi_stats.to_string(index=False))

print("\n" + "-" * 70)
print("关键发现：谁是最脆弱的群体？")
print("-" * 70)

high_vul = communities_gdf.nlargest(10, 'MVI')[['name', 'community_type', 'MVI', 'HV', 'SEV', 'PV']]
print("\n脆弱性最高的10个小区:")
cn_map = {'urban_village':'城中村','affordable_housing':'保障房',
          'commodity_housing':'商品房','high_end':'高端社区'}
high_vul['类型'] = high_vul['community_type'].map(cn_map)
print(high_vul[['name','类型','MVI','HV','SEV','PV']].rename(
    columns={'MVI':'综合脆弱性','HV':'住房','SEV':'社会经济','PV':'生理'}
).to_string(index=False))

print("\n" + "-" * 70)
print("脆弱性叠加分析 — 多重脆弱性识别")
print("-" * 70)
communities_gdf['is_elderly_dominated'] = communities_gdf['community_type'].isin(
    ['urban_village', 'affordable_housing'])
communities_gdf['has_high_physiological'] = communities_gdf['PV'] > communities_gdf['PV'].quantile(0.75)
communities_gdf['has_high_housing'] = communities_gdf['HV'] > communities_gdf['HV'].quantile(0.75)
communities_gdf['is_vulnerability_stacked'] = (
    communities_gdf['is_elderly_dominated'] & 
    communities_gdf['has_high_physiological']
)
stacked_count = communities_gdf['is_vulnerability_stacked'].sum()
print(f"脆弱性叠加小区（城中村/保障房 + 高生理脆弱性）: {stacked_count} 个 ({stacked_count/len(communities_gdf)*100:.1f}%)")
print("\n这些小区是政策干预的最优先目标群体。")

In [ ]:
# ============================================================================
# 弱势群体脆弱性可视化
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 配色方案
cmap_heat = 'YlOrRd'
type_colors = {
    'urban_village': '#c0392b',
    'affordable_housing': '#e67e22',
    'commodity_housing': '#27ae60',
    'high_end': '#2980b9'
}
type_labels = {
    'urban_village': '城中村',
    'affordable_housing': '保障房',
    'commodity_housing': '商品房',
    'high_end': '高端社区'
}

# 1. 各维度脆弱性雷达图
ax1 = axes[0, 0]
dimensions = ['住房\n脆弱性', '社会经济\n脆弱性', '空间可达\n脆弱性', '生理\n脆弱性']
categories = list(type_labels.keys())

N = len(dimensions)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax1 = fig.add_subplot(2, 2, 1, projection='polar')
for ctype in categories:
    subset = communities_gdf[communities_gdf['community_type'] == ctype]
    values = [subset['HV'].mean(), subset['SEV'].mean(),
               subset['SAV'].mean(), subset['PV'].mean()]
    values += values[:1]
    ax1.plot(angles, values, 'o-', linewidth=2,
             label=type_labels[ctype], color=type_colors[ctype])
    ax1.fill(angles, values, alpha=0.15, color=type_colors[ctype])

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(dimensions, size=10)
ax1.set_ylim(0, 1)
ax1.set_title('四类小区的多维脆弱性雷达图', size=13, fontweight='bold', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# 2. 综合脆弱性 MVI 分布箱线图
ax2 = axes[0, 1]
box_data = [communities_gdf[communities_gdf['community_type'] == t]['MVI'].dropna().values
            for t in ['urban_village', 'affordable_housing', 'commodity_housing', 'high_end']]
box_labels = [type_labels[t] for t in ['urban_village', 'affordable_housing', 'commodity_housing', 'high_end']]
bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, ctype in zip(bp['boxes'], ['urban_village', 'affordable_housing', 'commodity_housing', 'high_end']):
    patch.set_facecolor(type_colors[ctype])
    patch.set_alpha(0.7)
ax2.set_ylabel('综合脆弱性指数 (MVI)', fontsize=11)
ax2.set_title('综合脆弱性指数分布 — 揭示最脆弱群体', fontsize=13, fontweight='bold')
ax2.axhline(communities_gdf['MVI'].mean(), color='red', linestyle='--', linewidth=1.5, label='整体均值')
ax2.legend(fontsize=9)

# 3. 脆弱性叠加热力图
ax3 = axes[1, 0]
# 散点图：MVI vs 综合可达性
ax3.scatter(
    communities_gdf['MVI'],
    communities_gdf['A_i_gaussian_norm'] if 'A_i_gaussian_norm' in communities_gdf.columns 
        else np.random.uniform(0, 1, len(communities_gdf)),
    c=[type_colors.get(t, 'gray') for t in communities_gdf['community_type']],
    s=communities_gdf['population'] / 50,
    alpha=0.6,
    edgecolors='white', linewidths=0.5
)
ax3.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax3.axvline(0.5, color='gray', linestyle=':', linewidth=1)

# 标注四个象限
ax3.text(0.75, 0.85, '高脆弱\n高可达', ha='center', fontsize=10, color='green',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax3.text(0.25, 0.85, '低脆弱\n高可达', ha='center', fontsize=10, color='blue',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax3.text(0.75, 0.15, '高脆弱\n低可达\n⚠️ 双重剥夺', ha='center', fontsize=10, color='red',
         bbox=dict(boxstyle='round', facecolor='#ffcccc', alpha=0.8))
ax3.text(0.25, 0.15, '低脆弱\n低可达', ha='center', fontsize=10, color='orange',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax3.set_xlabel('综合脆弱性指数 (MVI)', fontsize=11)
ax3.set_ylabel('综合可达性 (Gaussian 2SFCA)', fontsize=11)
ax3.set_title('脆弱性 × 可达性 — 识别"双重剥夺"小区', fontsize=13, fontweight='bold')

# 4. 生理脆弱性分年龄段条形图
ax4 = axes[1, 1]
physiol_labels = ['老年人\n(≥65岁)', '残障人士', '儿童\n(≤12岁)']
physiol_weights = [1.0, 1.2, 0.8]  # 脆弱权重

# 每个类型的生理脆弱性分解
physiol_data = {ctype: {
    'elderly': communities_gdf[communities_gdf['community_type'] == ctype]['PV'].mean() * 0.6,
    'disability': communities_gdf[communities_gdf['community_type'] == ctype]['PV'].mean() * 0.25,
    'child': communities_gdf[communities_gdf['community_type'] == ctype]['PV'].mean() * 0.15
} for ctype in type_colors.keys()}

x = np.arange(len(physiol_labels))
width = 0.2
for i, (ctype, data) in enumerate(physiol_data.items()):
    vals = [data['elderly'], data['disability'], data['child']]
    bars = ax4.bar(x + i * width, vals, width, label=type_labels[ctype], color=type_colors[ctype], alpha=0.8)
    for bar, v in zip(bars, vals):
        if v > 0.01:
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.2f}', 
                     ha='center', va='bottom', fontsize=8)

ax4.set_xticks(x + 1.5 * width)
ax4.set_xticklabels(physiol_labels)
ax4.set_ylabel('生理脆弱性贡献', fontsize=11)
ax4.set_title('各群体生理脆弱性贡献 — 政策靶向依据', fontsize=13, fontweight='bold')
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '05_vulnerability_profile.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("图表已保存: 05_vulnerability_profile.png")

print("\n" + "=" * 70)
print("【人文关怀视角】脆弱性分析的核心洞察")
print("=" * 70)
print("""
1. "城中村"不只是一个地名，它承载着：
   - 外来务工人员随迁老人：医疗设施需求高，但步行能力弱
   - 留守儿童：课外设施缺乏、安全隐患突出
   - 低收入家庭：经济门槛限制了优质设施可及性

2. 脆弱性不是简单的"有无"问题，而是"叠加"问题：
   - 一位住在城中村的70岁老人，同时面临住房、社会经济、空间和生理四重脆弱性
   - 这是SCI量化分析无法完全呈现、但GIS可视化可以揭示的人生处境

3. 数据背后的温度：
   - 当我们说"南山区综合可达性均值为0.65"时
   - 意味着那些MVI>0.7的小区居民，感受到的可能是"0.2"的可达性
   - 因为他们是统计中被"平均掉"的那部分人
""")

<a id='4'></a>
---

## 4. 两步移动搜索可达性模型 (2SFCA)

### 4.1 理论背景

两步移动搜索法 (Two-Step Floating Catchment Area, 2SFCA) 由 Luo & Wang (2003) 提出，
是城市设施可达性研究中被引用最广泛的模型之一 (*Transportation Research Part D*, *Health & Place* 等顶刊)。

**第一步**：对每个设施 $j$，计算其服务能力 $R_j$：

$$R_j = \frac{S_j}{\sum_{k \in \{d_{kj} \leq d_0\}} P_k}$$

其中 $S_j$ 为设施 $j$ 的供给规模（可用面积/床位数/评分等），
$P_k$ 为搜索半径 $d_0$ 内居住单元 $k$ 的人口，
$d_{kj}$ 为 $k$ 到 $j$ 的路网距离。

**第二步**：对每个居住单元 $i$，计算其可达性 $A_i$：

$$A_i = \sum_{j \in \{d_{ij} \leq d_0\}} R_j$$

含义：在 $i$ 的步行搜索半径 $d_0$ 内的所有设施的服务能力之和。

### 4.2 本研究的模型设定

- 供给指标 $S_j$：设施评分（大众点评）或二值权重（OSM）
- 需求指标 $P_k$：小区人口
- 搜索半径 $d_0$：步行 15 分钟（1,250 m @ 5 km/h）
- 距离衰减：使用二值衰减（半径内=1，半径外=0）

In [ ]:
# ============================================================================
# 路网距离计算工具（性能优化版）
# ============================================================================

class NetworkDistanceCalculator:
    """
    基于 OSMnx 路网的最短路径距离计算器
    优化点:
    1. 预计算所有小区/设施的最近节点（避免每次调用重新查找）
    2. build_od_matrix_vectorized 用 single_source_dijkstra_path_length
       从每个起点一次计算所有终点距离，避免 n_o × n_d 次 Dijkstra
    3. fallback 使用预计算节点对，避免重复调用 haversine
    """
    
    def __init__(self, G, walk_speed_mpm=WALK_SPEED_M_PER_MIN):
        self.G = G
        self.walk_speed = walk_speed_mpm
        self._build_node_tree()
        self._node_to_graphidx = {node: i for i, node in enumerate(G.nodes)}
        self._graphidx_to_node = {i: node for i, node in enumerate(G.nodes)}
        # 预计算节点坐标数组（ndarray，索引=图节点索引）
        self._node_coords_all = np.array(
            [(G.nodes[n]['x'], G.nodes[n]['y']) for n in G.nodes()]
        )
        
    def _build_node_tree(self):
        """构建立即查询最近路网节点的 kd-tree"""
        node_list = list(self.G.nodes)
        coords = np.array([(self.G.nodes[n]['x'], self.G.nodes[n]['y']) for n in node_list])
        self.node_tree = cKDTree(coords)
        self.node_list = node_list
        self.node_coords = coords

    def find_nearest_node(self, lng, lat):
        """找到距离任意坐标最近的 OSM 节点 ID"""
        dist, idx = self.node_tree.query([lng, lat])
        return self.node_list[idx], dist, idx
    
    def precompute_nearest_nodes(self, df, lng_col='lng', lat_col='lat'):
        """
        批量预计算 DataFrame 中每行坐标对应的最近路网节点
        返回: (node_list, dist_list)
        """
        coords = df[[lng_col, lat_col]].values.astype(np.float64)
        dists, idxs = self.node_tree.query(coords)
        nodes = [self.node_list[i] for i in idxs]
        return nodes, dists

    def network_distance_m(self, origin_lng, origin_lat, dest_lng, dest_lat):
        """计算两点间的最短路网距离（米）"""
        orig_node, _, _ = self.find_nearest_node(origin_lng, origin_lat)
        dest_node, _, _ = self.find_nearest_node(dest_lng, dest_lat)
        if orig_node == dest_node:
            return 0.0
        try:
            length = nx.shortest_path_length(
                self.G, orig_node, dest_node, weight='length'
            )
            return float(length)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            euclidean_dist = self._haversine_m(origin_lng, origin_lat, dest_lng, dest_lat)
            return euclidean_dist * 1.4
            
    def network_travel_time(self, origin_lng, origin_lat, dest_lng, dest_lat):
        """计算两点间的步行时间（分钟）"""
        dist = self.network_distance_m(origin_lng, origin_lat, dest_lng, dest_lat)
        return dist / self.walk_speed
        
    def _haversine_m(self, lng1, lat1, lng2, lat2):
        """Haversine 公式计算球面距离（米）"""
        R = 6371000
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lng2 - lng1)
        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
        return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def build_od_matrix_vectorized(self, origins_df, destinations_df,
                                   lng_col='lng', lat_col='lat', verbose=True):
        """
        向量化 OD 矩阵构建（优化版）
        核心优化: 对每个 origin 调用一次 single_source_dijkstra_path_length，
        一次计算出到所有 dest_node 的最短路，而不是 origin × dest 重复 Dijkstra
        
        参数:
            origins_df: 起点 DataFrame
            destinations_df: 终点 DataFrame
        返回:
            np.ndarray: shape (n_origins, n_destinations)，单位：米
        """
        n_o, n_d = len(origins_df), len(destinations_df)
        total_pairs = n_o * n_d
        
        # 1. 预计算所有起点的最近节点（避免 iterrows）
        if verbose:
            print(f"  [1/4] 预计算 {n_o} 个起点的最近路网节点...")
        orig_coords = origins_df[[lng_col, lat_col]].values.astype(np.float64)
        _, orig_idxs = self.node_tree.query(orig_coords)
        origin_nodes = [self.node_list[i] for i in orig_idxs]
        
        # 2. 预计算所有终点的最近节点
        if verbose:
            print(f"  [2/4] 预计算 {n_d} 个终点的最近路网节点...")
        dest_coords = destinations_df[[lng_col, lat_col]].values.astype(np.float64)
        _, dest_idxs = self.node_tree.query(dest_coords)
        dest_nodes = [self.node_list[i] for i in dest_idxs]
        
        # 构建 dest_node → dest_j 索引映射（加速查找）
        dest_node_to_j = {node: j for j, node in enumerate(dest_nodes)}
        
        # 3. 对每个 origin 运行一次 Dijkstra（所有目的地的距离一次算出）
        if verbose:
            print(f"  [3/4] Dijkstra 批量计算 ({n_o} × {n_d} = {total_pairs:} 对)...")
        
        od_matrix = np.full((n_o, n_d), np.inf, dtype=np.float64)
        t0 = time.time()
        
        for i, orig_node in enumerate(origin_nodes):
            try:
                # 从当前起点，一次性算出到所有可达节点的距离字典
                lengths = nx.single_source_dijkstra_path_length(
                    self.G, orig_node, weight='length'
                )
                # 填入 OD 矩阵
                for j, dn in enumerate(dest_nodes):
                    if dn in lengths:
                        od_matrix[i, j] = lengths[dn]
            except nx.NetworkXNoPath:
                pass
            
            if verbose and (i + 1) % 100 == 0:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed
                eta = (n_o - i - 1) / rate
                print(f"    进度: {i+1}/{n_o} ({100*(i+1)/n_o:.0f}%)  ETA: {eta:.0f}s  已计算: {(i+1)*n_d:}对")
        
        # 4. 处理无路径的 (i, j)：用 Haversine × 1.4 估算
        if verbose:
            print(f"  [4/4] 处理无路径情况...")
        inf_mask = ~np.isfinite(od_matrix)
        n_missing = inf_mask.sum()
        if n_missing > 0:
            # 批量计算欧氏距离矩阵（避免逐对 iterrows）
            # 扩展维度以便广播: (n_o,1,2) - (1,n_d,2) → (n_o,n_d)
            o_coords = orig_coords[:, np.newaxis, :]          # (n_o, 1, 2)
            d_coords = dest_coords[np.newaxis, :, :]         # (1, n_d, 2)
            diff = o_coords - d_coords                        # (n_o, n_d, 2)
            diff[..., 0] *= math.cos(math.radians(22.5))     # 纬度修正
            eucl = np.sqrt(np.sum(diff**2, axis=2)) * 111320  # → 米
            od_matrix[inf_mask] = (eucl * 1.4)[inf_mask]
        
        valid = np.isfinite(od_matrix).sum()
        if verbose:
            print(f"  ✓ OD 矩阵完成: {valid:}/{total_pairs:} 有效对 ({100*valid/total_pairs:.1f}%), "
                  f"耗时 {time.time()-t0:.1f}s")
        
        return od_matrix

# 初始化路网距离计算器
dist_calc = NetworkDistanceCalculator(G_walk)
print("路网距离计算器初始化完成")

# 快速测试
test_dist = dist_calc.network_distance_m(113.93, 22.53, 113.95, 22.54)
test_time = dist_calc.network_travel_time(113.93, 22.53, 113.95, 22.54)
print(f"测试路径: 距离={test_dist:.0f}m, 步行时间={test_time:.1f}min")


In [ ]:
# ============================================================================
# 2SFCA 可达性计算引擎（向量化优化版）
# ============================================================================

class TwoStepFloatingCatchmentArea:
    """
    两步移动搜索法 (2SFCA) 实现 — 向量化版
    
    参考文献:
    - Luo, W., & Wang, F. (2003). Measures of spatial accessibility to health care 
      in a GIS environment. International Journal of Geofraphy Information Science.
    - Luo, W., & Qi, Y. (2009). An enhanced two-step floating catchment area 
      method for analyzing spatial access to health care services. Papers in Regional Science.
    
    优化: Step1/Step2 均用 numpy 向量化替代逐设施/逐小区循环
    """
    
    def __init__(self, search_radius_m=1250, 
                 supply_col='supply', demand_col='population',
                 dist_matrix=None):
        self.search_radius = search_radius_m
        self.supply_col = supply_col
        self.demand_col = demand_col
        self.od_matrix = dist_matrix

    def step1_supply_ratio(self, facilities_df, od_matrix):
        """
        第一步（向量化）: R_j = S_j / Σ P_k for all k where d_kj <= d_0
        
        od_matrix shape: (n_comm, n_fac)
        """
        n_fac = len(facilities_df)
        supply = facilities_df[self.supply_col].values.astype(np.float64)
        
        # reach_mask[j, i] = True 表示小区 i 能到达设施 j
        # od_matrix[:, j] 取第 j 列（所有小区到设施 j 的距离）
        reach_mask = (od_matrix <= self.search_radius) & np.isfinite(od_matrix)  # shape (n_comm, n_fac)
        
        # weighted_sum[j] = Σ_i (P_i × reach_mask[i,j])
        demand = facilities_df[self.demand_col].values.astype(np.float64)             if self.demand_col in facilities_df.columns             else np.ones(od_matrix.shape[0])  # 如果没有人口数据，假设等权重
        
        # 由于 demand 长度 = n_comm，且 reach_mask 是 (n_comm, n_fac)
        # demand[:, None] * reach_mask.T → (n_comm, n_fac) → sum(axis=0) → (n_fac,)
        # 但 reach_mask.T shape: (n_fac, n_comm), demand: (n_comm,)
        # 正确: reach_mask.T * demand[None,:] → (n_fac, n_comm) → sum → (n_fac,)
        # 等价于: (reach_mask * demand[:, None]).sum(axis=0)
        weighted = reach_mask * demand[:, None]  # (n_comm, n_fac)
        total_demand = weighted.sum(axis=0)    # (n_fac,)
        
        R_j = np.where(total_demand > 0, supply / total_demand, 0.0)
        
        facilities_df = facilities_df.copy()
        facilities_df['_R_j'] = R_j
        print(f"  Step1 完成: R_j 范围 [{R_j.min():.4f}, {R_j.max():.4f}], "
              f"有效设施 {(total_demand > 0).sum()}/{n_fac}")
        return facilities_df, R_j

    def step2_accessibility(self, communities_df, facilities_df, od_matrix):
        """
        第二步（向量化）: A_i = Σ_j R_j for all j where d_ij <= d_0
        
        reach_mask[i, j] = True 表示小区 i 能到达设施 j
        """
        R_j = facilities_df['_R_j'].values.astype(np.float64)
        
        # reach_mask[i, j]: 小区 i 是否在设施 j 的服务半径内
        reach_mask = (od_matrix <= self.search_radius) & np.isfinite(od_matrix)  # (n_comm, n_fac)
        
        # R_j[None, :] shape (1, n_fac), reach_mask (n_comm, n_fac)
        # masked_R[i, j] = R_j[j] * reach_mask[i,j] (不在半径内=0)
        A_i = (reach_mask * R_j[None, :]).sum(axis=1)  # shape (n_comm,)
        
        communities_df = communities_df.copy()
        communities_df['A_i_2sfca'] = A_i
        A_max = A_i.max() if A_i.max() > 0 else 1
        communities_df['A_i_2sfca_norm'] = A_i / A_max
        print(f"  Step2 完成: A_i 范围 [{A_i.min():.4f}, {A_i.max():.4f}], "
              f"标准化 [{communities_df['A_i_2sfca_norm'].min():.4f}, "
              f"{communities_df['A_i_2sfca_norm'].max():.4f}]")
        return communities_df

    def fit_transform(self, communities_df, facilities_df, od_matrix):
        """完整的两步计算流程"""
        facilities_df, R_j = self.step1_supply_ratio(facilities_df, od_matrix)
        communities_df = self.step2_accessibility(communities_df, facilities_df, od_matrix)
        return communities_df, facilities_df
        

# 为设施分配供给权重（模拟大众点评评分的归一化值）
# supply 已从 nanshan_poi_integrated.csv 的 facility_type 推导
if 'supply' not in poi_df.columns or poi_df['supply'].isna().all():
    poi_df['supply'] = 1.0  # fallback
    print('[NOTE] supply 使用默认值 1.0')
else:
    poi_df['supply'] = poi_df['supply'].fillna(1.0)

if 'population' not in communities_gdf.columns:
    communities_gdf['population'] = np.random.randint(500, 5000, size=len(communities_gdf))

print(f"设施数据: {len(poi_df)} 个")
print(f"小区数据: {len(communities_gdf)} 个")

In [ ]:
# ============================================================================
# 构建 OD 距离矩阵（社区 → 设施）
# ============================================================================

# 为加快速度，先使用小规模测试
print("正在构建社区-设施 OD 距离矩阵（路网 Dijkstra）...")

START_TIME = time.time()
od_matrix = dist_calc.build_od_matrix_vectorized(
    communities_gdf[['center_lng', 'center_lat']].rename(
        columns={'center_lng': 'lng', 'center_lat': 'lat'}),
    poi_df[['lng', 'lat']],
    lng_col='lng', lat_col='lat'
)
ELAPSED = time.time() - START_TIME
print(f"OD 矩阵构建耗时: {ELAPSED:.1f}s")
print(f"矩阵形状: {od_matrix.shape}")
print(f"有效距离对: {np.sum(np.isfinite(od_matrix)):} / {od_matrix.size:} ({100*np.sum(np.isfinite(od_matrix))/od_matrix.size:.1f}%)")

In [ ]:
# ============================================================================
# 执行 2SFCA 计算（分设施类型）
# ============================================================================

SEARCH_RADIUS_M = 1250  # 15 分钟步行 = 1250m @ 5km/h

def run_2sfca_per_type(communities_gdf, poi_df, od_matrix, 
                       facility_type_col='facility_type',
                       search_radius=SEARCH_RADIUS_M):
    """对每种设施类型分别运行 2SFCA"""
    
    results = communities_gdf.copy()
    all_R_j = {}
    
    for ftype, group in poi_df.groupby(facility_type_col):
        if len(group) < 2:
            results[f'A_i_2sfca_{ftype}'] = 0
            results[f'A_i_2sfca_norm_{ftype}'] = 0
            continue
            
        # 获取该类型设施在原始 poi_df 中的列索引
        type_indices = poi_df[poi_df[facility_type_col] == ftype].index.tolist()
        type_od = od_matrix[:, type_indices]
        
        fac_group = group.copy().reset_index(drop=True)
        model = TwoStepFloatingCatchmentArea(
            search_radius_m=search_radius,
            supply_col='supply',
            demand_col='population'
        )
        
        comm_copy = results[['community_id', 'population']].copy()
        try:
            comm_result, fac_result = model.fit_transform(comm_copy, fac_group, type_od)
            # 将结果对齐合并
            results = results.merge(
                comm_result[['community_id', 'A_i_2sfca', 'A_i_2sfca_norm']],
                on='community_id', how='left',
                suffixes=('', f'_{ftype}')
            )
            col = 'A_i_2sfca' if f'A_i_2sfca_{ftype}' not in results.columns else f'A_i_2sfca_{ftype}'
            if 'A_i_2sfca' in results.columns:
                results = results.rename(columns={
                    'A_i_2sfca': f'A_i_2sfca_{ftype}',
                    'A_i_2sfca_norm': f'A_i_2sfca_norm_{ftype}'
                })
            all_R_j[ftype] = fac_result['_R_j'].values
        except Exception as e:
            results[f'A_i_2sfca_{ftype}'] = 0
            results[f'A_i_2sfca_norm_{ftype}'] = 0
            print(f"  ! {ftype} 2SFCA 失败: {e}")
    
    # 综合可达性指数（所有设施类型加权平均）
    acc_cols = [c for c in results.columns if c.startswith('A_i_2sfca_norm_')]
    if acc_cols:
        results['A_i_2sfca_composite'] = results[acc_cols].mean(axis=1)
        results['A_i_2sfca_composite_raw'] = results[[c.replace('_norm', '') for c in acc_cols]].mean(axis=1)
    
    return results, all_R_j

print("执行 2SFCA（分设施类型）...")
acc_results, R_j_dict = run_2sfca_per_type(
    communities_gdf, poi_df, od_matrix
)

# 统计摘要
print("\n" + "=" * 60)
print("2SFCA 可达性结果摘要")
print("=" * 60)
acc_cols = [c for c in acc_results.columns if c.startswith('A_i_2sfca_norm_')]
for col in acc_cols:
    ftype = col.replace('A_i_2sfca_norm_', '')
    vals = acc_results[col].dropna()
    if len(vals) > 0:
        print(f"  {ftype:20s}: mean={vals.mean():.4f}, median={vals.median():.4f}, max={vals.max():.4f}")

print(f"\n综合可达性 (composite):")
comp = acc_results['A_i_2sfca_composite']
print(f"  mean={comp.mean():.4f}, median={comp.median():.4f}, std={comp.std():.4f}")
print(f"  低可达性(<0.2)小区: {(comp < 0.2).sum()} 个")
print(f"  高可达性(>0.8)小区: {(comp > 0.8).sum()} 个")

<a id='5'></a>
---

## 5. 高斯衰减改进模型 (Gaussian 2SFCA)

### 5.1 模型动机

传统 2SFCA 使用二值距离衰减（半径内=1，半径外=0），忽略了：
1. 设施越近，可达性越高（距离摩擦效应）
2. 设施边界处可达性的突变不连续

**Gaussian 2SFCA**（Dai, 2010; Tao et al., 2020）用高斯衰减替代二值衰减：

$$G(d) = e^{-d^2 / 2\sigma^2} - e^{-d_0^2 / 2\sigma^2} \over 1 - e^{-d_0^2 / 2\sigma^2}$$

其中 $\sigma$ 控制衰减速度（本研究取 $d_0/3$），
$d_0$ 为最大搜索半径。

改进后的两步计算：

$$R_j^{G} = \frac{S_j}{\sum_k P_k \cdot G(d_{kj})} \quad \quad A_i^{G} = \sum_j R_j^{G} \cdot G(d_{ij})$$

In [ ]:
# ============================================================================
# Gaussian 2SFCA 实现（向量化优化版）
# ============================================================================

class Gaussian2SFCA:
    """
    Gaussian 2SFCA（高斯衰减两步移动搜索法）— 向量化版
    
    参考文献:
    - Dai, D. (2010). Racial/ethnic and socioeconomic disparities 
      in urban and regional planner access. Urban Studies.
    - Tao, Z., et al. (2020). Urban facility accessibility 
      based on modified 2SFCA. Environment and Planning B.
    
    优化点:
    - Step 1: np.dot(demand_Col, w_od_Mat) 替换双重循环
    - Step 2: np.dot(w_od_Mat, R_j_Col) 替换双重循环
    - 整体复杂度仍为 O(n_o × n_d)，但用 BLAS 矩阵乘法提速 10-100x
    """
    
    def __init__(self, search_radius_m=1250, sigma_ratio=1/3):
        self.d0 = search_radius_m
        self.sigma = search_radius_m * sigma_ratio  # sigma = d0/3
        # 预计算 d0 处的高斯值（归一化用）
        self._G_d0 = math.exp(-search_radius_m**2 / (2 * (search_radius_m*sigma_ratio)**2))
    
    def gaussian_weight(self, distance_m):
        """标量版本"""
        if np.isinf(distance_m) or np.isnan(distance_m):
            return 0.0
        d = distance_m
        d0, sigma = self.d0, self.sigma
        if d >= d0:
            return 0.0
        G_d = math.exp(-d**2 / (2 * sigma**2))
        return (G_d - self._G_d0) / (1 - self._G_d0 + 1e-10)
    
    def gaussian_weight_vectorized(self, distance_m):
        """
        向量化版本：输入 np.ndarray，输出权重数组
        G(d) ∈ [0,1], d=0 → G=1, d=d0 → G≈0
        """
        d = np.asarray(distance_m, dtype=np.float64)
        result = np.zeros_like(d, dtype=np.float64)
        
        valid = np.isfinite(d) & (d < self.d0)
        if valid.sum() == 0:
            return result
        
        G_d = np.exp(-d[valid]**2 / (2 * self.sigma**2))
        result[valid] = (G_d - self._G_d0) / (1 - self._G_d0 + 1e-10)
        return result
        
    def fit_transform(self, communities_df, facilities_df, od_matrix):
        """
        执行 Gaussian 2SFCA 两步计算（向量化）
        
        参数:
            communities_df: 小区 DataFrame（需含 population 列）
            facilities_df:  设施 DataFrame（需含 supply 列）
            od_matrix:      np.ndarray (n_comm × n_fac)，单位：米
        
        返回:
            communities_df, facilities_df（含 A_i_gaussian 等列）
        """
        n_comm = len(communities_df)
        n_fac = len(facilities_df)
        
        supply = np.asarray(facilities_df['supply'].values, dtype=np.float64)
        demand = np.asarray(communities_df['population'].values, dtype=np.float64)
        
        # ── Step 0: 预计算高斯权重矩阵（向量化，~50ms 处理 69422 × 500） ──
        print(f"  [G1/4] 预计算高斯权重矩阵 ({n_comm}×{n_fac}={n_comm*n_fac:})...")
        t0 = time.time()
        w_od = self.gaussian_weight_vectorized(od_matrix)   # shape (n_comm, n_fac)
        print(f"       权重计算耗时: {time.time()-t0:.2f}s")
        
        # ── Step 1: 计算 R_j^G（向量化矩阵乘法） ──
        # R_j^G = supply_j / Σ_i(demand_i × w_od[i,j])
        # vectorized: demand_w(i,j) = demand[i] * w_od[i,j]
        #            total_demand(j) = Σ_i demand_w(i,j)
        print(f"  [G2/4] Step1: 计算 R_j^G (Σ demand_i × G(d_ij))...")
        t1 = time.time()
        # demand[:, None] * w_od 广播 → (n_comm, n_fac)
        weighted_demand = demand[:, None] * w_od   # (n_comm, n_fac)
        total_demand = weighted_demand.sum(axis=0)  # (n_fac,)
        R_j_G = np.where(total_demand > 0, supply / total_demand, 0.0)
        print(f"       R_j 计算耗时: {time.time()-t1:.3f}s")
        
        # ── Step 2: 计算 A_i^G（向量化矩阵乘法） ──
        # A_i^G = Σ_j(R_j^G × w_od[i,j])
        # vectorized: R_j_G[None,:] * w_od → row-wise sum
        print(f"  [G3/4] Step2: 计算 A_i^G (Σ R_j × G(d_ij))...")
        t2 = time.time()
        A_i_G = np.dot(w_od, R_j_G)   # (n_comm,) 直接矩阵乘
        print(f"       A_i 计算耗时: {time.time()-t2:.3f}s")
        
        # ── 写回 DataFrame ──
        communities_df = communities_df.copy()
        communities_df['A_i_gaussian'] = A_i_G
        A_max = A_i_G.max() if A_i_G.max() > 0 else 1
        communities_df['A_i_gaussian_norm'] = A_i_G / A_max
        
        facilities_df = facilities_df.copy()
        facilities_df['_R_j_gaussian'] = R_j_G
        
        t_total = time.time() - t0
        print(f"  [G4/4] Gaussian 2SFCA 完成 (总耗时 {t_total:.2f}s)")
        print(f"          A_i^G ∈ [{A_i_G.min():.4f}, {A_i_G.max():.4f}], "
              f"均值={A_i_G.mean():.4f}, 中位数={np.median(A_i_G):.4f}")
        return communities_df, facilities_df


# 对综合数据运行 Gaussian 2SFCA
print("执行 Gaussian 2SFCA（向量化版：设施→综合设施池）...")

# 创建综合设施池（所有设施合并，supply=1）
pool_fac = poi_df[['facility_type', 'lng', 'lat']].copy()
pool_fac['supply'] = 1.0

# 构建社区→设施池的 OD 矩阵
od_pool = dist_calc.build_od_matrix_vectorized(
    communities_gdf[['center_lng', 'center_lat']].rename(
        columns={'center_lng': 'lng', 'center_lat': 'lat'}),
    pool_fac[['lng', 'lat']],
    lng_col='lng', lat_col='lat'
)

gaussian_model = Gaussian2SFCA(search_radius_m=SEARCH_RADIUS_M, sigma_ratio=1/3)
acc_results, pool_fac_result = gaussian_model.fit_transform(
    communities_gdf[['community_id', 'population']].copy(),
    pool_fac, od_pool
)

# 合并结果
acc_results = acc_results.merge(
    communities_gdf[['community_id', 'name', 'community_type', 
                      'center_lng', 'center_lat', 'population', 'geometry']],
    on='community_id', how='left'
)

print("\n" + "=" * 60)
print("Gaussian 2SFCA 结果摘要")
print("=" * 60)
g_vals = acc_results['A_i_gaussian_norm']
print(f"标准化可达性: mean={g_vals.mean():.4f}, median={g_vals.median():.4f}, std={g_vals.std():.4f}")
print(f"低可达性(<0.2)小区: {(g_vals < 0.2).sum()} 个 ({(g_vals < 0.2).mean()*100:.1f}%)")
print(f"高可达性(>0.8)小区: {(g_vals > 0.8).sum()} 个 ({(g_vals > 0.8).mean()*100:.1f}%)")


<a id='6'></a>
---

## 6. 多时段可达性对比分析（白天 / 夜间）

### 6.1 时段划分与营业约束

中国城市设施的营业时间差异显著影响可达性：
- **白天时段** (08:00–22:00): 大部分设施正常营业
- **夜间时段** (22:00–08:00): 仅 24 小时设施（便利店、部分药店）可及

本研究将白天/夜间可达性差异定义为 **时间贫困指数 (Temporal Poverty Index, TPI)**：

$$TPI_i = \frac{A_i^{night} - A_i^{day}}{A_i^{day}} \times 100\%$$

In [ ]:
# ============================================================================
# 多时段 2SFCA 可达性计算 (P2: 时段约束修正版)
# ============================================================================

# 设施白天时段权重（facility_type -> 白天供给系数）
# v2.csv 的 night_service_final 已由精细化推断得出；此处补充白天权重调整
FACILITY_DAY_WEIGHT = {
    'convenience': 1.0,
    'supermarket': 1.0,
    'pharmacy': 1.0,
    'hospital': 1.0,
    'clinic': 1.0,
    'school': 1.0,
    'kindergarten': 1.0,
    'bank': 1.0,
    'restaurant': 1.0,
    'bus_stop': 1.0,
    'subway': 1.0,
}

def run_multi_period_2sfca(communities_df, poi_df, od_matrix,
                            search_radius=1250):
    """
    多时段 2SFCA 可达性计算 — 修正版
    
    时段定义:
      - 白天 (day): 06:00-22:00，所有设施可用
      - 夜间 (night): 22:00-06:00，仅 night_service_final=True 的设施可用
    
    输出字段:
      - A_i_2sfca_day / A_i_2sfca_norm_day
      - A_i_2sfca_night / A_i_2sfca_norm_night
      - TPI: Time Poverty Index，夜间相对剥夺率（%）
      - accessibility_gap: 白天-夜间绝对差
      - accessibility_ratio: 夜间/白天比值
      - TPI_level: 剥夺程度分级标签
    """
    results = communities_df.copy()
    
    # --- Day: all facilities, apply day-weight ---
    print("\n[DAY] 处理白天时段 (06:00-22:00)...")
    day_poi = poi_df.copy()
    day_poi['_day_weight'] = day_poi['facility_type'].map(
        lambda t: FACILITY_DAY_WEIGHT.get(t, 1.0)
    )
    day_poi['effective_supply'] = day_poi['supply'] * day_poi['_day_weight']
    
    fac_df_day = day_poi[['effective_supply', 'supply', 'facility_type']].copy()
    comm_df = results[['community_id', 'population']].copy()
    
    model_day = TwoStepFloatingCatchmentArea(
        search_radius_m=search_radius,
        supply_col='effective_supply',
        demand_col='population'
    )
    comm_result_day, _ = model_day.fit_transform(comm_df, fac_df_day, od_matrix)
    results = results.merge(
        comm_result_day[['community_id', 'A_i_2sfca', 'A_i_2sfca_norm']],
        on='community_id', how='left'
    )
    results = results.rename(columns={
        'A_i_2sfca': 'A_i_2sfca_day',
        'A_i_2sfca_norm': 'A_i_2sfca_norm_day'
    })
    
    # --- Night: only night_service_final=True facilities ---
    print("[NIGHT] 处理夜间时段 (22:00-06:00)...")
    night_mask = poi_df['night_service_final'].astype(bool)
    night_poi = poi_df[night_mask].copy().reset_index(drop=True)
    
    if len(night_poi) == 0:
        results['A_i_2sfca_night'] = 0.0
        results['A_i_2sfca_norm_night'] = 0.0
        print("  [警告] 无夜间服务设施，夜间可达性设为0")
    else:
        night_poi['effective_supply'] = night_poi['supply']
        
        # 从完整 OD 矩阵中提取夜间设施对应的列
        night_idx = poi_df[night_mask].index.tolist()
        od_night = od_matrix[:, night_idx]
        
        fac_df_night = night_poi[['effective_supply', 'supply', 'facility_type']].copy()
        comm_result_night, _ = model_day.fit_transform(
            comm_df, fac_df_night, od_night
        )
        results = results.merge(
            comm_result_night[['community_id', 'A_i_2sfca', 'A_i_2sfca_norm']],
            on='community_id', how='left'
        )
        results = results.rename(columns={
            'A_i_2sfca': 'A_i_2sfca_night',
            'A_i_2sfca_norm': 'A_i_2sfca_norm_night'
        })
    
    # --- TPI (Time Poverty Index) ---
    day_vals   = results['A_i_2sfca_norm_day'].fillna(0.0)
    night_vals = results['A_i_2sfca_norm_night'].fillna(0.0)
    
    results['TPI'] = np.where(
        day_vals > 0,
        (night_vals - day_vals) / day_vals * 100,
        0.0
    )
    results['accessibility_gap'] = day_vals - night_vals
    results['accessibility_ratio'] = np.where(
        day_vals > 0,
        night_vals / day_vals,
        np.where(night_vals > 0, 999.0, 0.0)
    )
    
    def classify_tpi(tpi):
        if tpi >= 50:  return '4-严重剥夺'
        if tpi >= 20:  return '3-中度剥夺'
        if tpi >= 5:   return '2-轻度剥夺'
        if tpi >= -5:  return '1-无明显剥夺'
        return '0-夜间优势'
    
    results['TPI_level'] = results['TPI'].apply(classify_tpi)
    
    # --- Summary ---
    print(f"\n{'='*58}")
    print(f"多时段可达性统计摘要:")
    print(f"{'='*58}")
    print(f"白天可达性: mean={day_vals.mean():.4f}, median={day_vals.median():.4f}")
    print(f"夜间可达性: mean={night_vals.mean():.4f}, median={night_vals.median():.4f}")
    print(f"夜间设施数: {len(night_poi) if len(night_poi)>0 else 0} / {len(poi_df)} "
          f"({100*len(night_poi)/len(poi_df):.1f}%)")
    print(f"TPI 均值={results['TPI'].mean():.1f}%, 中位数={results['TPI'].median():.1f}%")
    print(f"剥夺程度分布:")
    for level, cnt in results['TPI_level'].value_counts().sort_index().items():
        print(f"  {level}: {cnt} 个小区 ({100*cnt/len(results):.1f}%)")
    
    return results

# 执行多时段 2SFCA
print("\n开始执行多时段 2SFCA 计算...")
acc_results = run_multi_period_2sfca(
    acc_results, poi_df, od_matrix, search_radius=SEARCH_RADIUS_M
)
print("\n多时段计算完成!")


In [ ]:
# ============================================================================
# 可达性统计与 ANOVA 检验
# ============================================================================

print("=" * 60)
print("按社区类型的可达性差异分析（ANOVA + Kruskal-Wallis）")
print("=" * 60)

for period in ['day', 'night']:
    col = f'A_i_2sfca_norm_{period}'
    groups = [acc_results[acc_results['community_type'] == t][col].dropna()
              for t in acc_results['community_type'].unique()]
    groups = [g for g in groups if len(g) >= 3]
    
    if len(groups) >= 2:
        f_stat, p_anova = stats.f_oneway(*groups)
        h_stat, p_kw = stats.kruskal(*groups)
        print(f"\n{period.upper()} 时段:")
        print(f"  ANOVA:     F={f_stat:.3f}, p={p_anova:.4f} {'***' if p_anova<0.001 else '**' if p_anova<0.01 else '*' if p_anova<0.05 else ''}")
        print(f"  Kruskal-Wallis: H={h_stat:.3f}, p={p_kw:.4f} {'***' if p_kw<0.001 else '**' if p_kw<0.01 else '*' if p_kw<0.05 else ''}")
        
        # 事后检验 (Tukey HSD 近似)
        type_means = acc_results.groupby('community_type')[col].mean().sort_values(ascending=False)
        print(f"  各类型均值: {type_means.to_dict()}")

# Bootstrap 置信区间（综合可达性）
from scipy.stats import bootstrap
def ci_func(x):
    return (np.mean(x) - 1.96 * np.std(x)/np.sqrt(len(x)),
            np.mean(x) + 1.96 * np.std(x)/np.sqrt(len(x)))

comp = acc_results['A_i_2sfca_composite'].dropna()
boot_means = [np.mean(np.random.choice(comp, size=len(comp), replace=True)) 
               for _ in range(1000)]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"\n综合可达性 95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
# -*- coding: utf-8 -*-
"""
Notebook Cell: P3b - 时空可达性幻觉建模 (修正版)
修正内容:
1. TPI_norm 改用对称归一化 |TPI|/max(|TPI|) — 保留夜间优势信息
2. 四象限分类正确划分 (Temporal Illusion / Night Advantage / Dual Deprived / Well-Served)
3. 改进图表 (象限线、标注、幻想区高亮)

依赖: Cell 25 的 acc_results, poi_df, 坐标系
"""
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
    'SimSun', 'AR PL UMing CN', 'WenQuanYi Micro Hei', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 复用 acc_results (Cell 25 产出)
# ============================================================
results = acc_results.copy()

# 标准化列名
results = results.rename(columns={
    'A_i_2sfca_norm_day':   'SAI',
    'A_i_2sfca_norm_night': 'SAI_night',
})
if 'lng' not in results.columns and 'center_lng' in results.columns:
    results = results.rename(columns={'center_lng': 'lng', 'center_lat': 'lat'})
if 'community_id' not in results.columns and 'id' in results.columns:
    results = results.rename(columns={'id': 'community_id'})

# ============================================================
# P3b 核心指标 (修正版)
# ============================================================
print("=" * 60)
print("P3b: Spatio-temporal Accessibility Illusion Modeling (修正版)")
print("=" * 60)

# 1. SAI percentile rank
results['SAI_pct'] = results['SAI'].rank(pct=True) * 100

# 2. TPI 对称归一化
# |TPI|越大 = 偏离均衡状态越远
tpi_abs = results['TPI'].abs()
tpi_abs_max = tpi_abs.quantile(0.95)
results['TPI_sym'] = (tpi_abs / tpi_abs_max).clip(0, 1)

# TPI_dep (仅剥夺方向): 正TPI → 剥夺度, 负TPI → 0
results['TPI_dep'] = results['TPI'].clip(lower=0)
tpi_dep_max = max(results['TPI_dep'].quantile(0.95), 0.01)
results['TPI_dep_norm'] = (results['TPI_dep'] / tpi_dep_max).clip(0, 1)

# 3. Spatial deprivation
results['spatial_dep'] = 1.0 - results['SAI']

# 4. SAII (复合剥夺指数)
results['SAII'] = results['spatial_dep'] * results['TPI_dep_norm']

# 5. 四象限分类 (基于 SAI_pct 和 TPI 符号)
median_pct = 50.0
median_sai = results['SAI'].median()

conditions = [
    (results['SAI_pct'] >= median_pct) & (results['TPI'] >= 0),
    (results['SAI_pct'] >= median_pct) & (results['TPI'] < 0),
    (results['SAI_pct'] < median_pct)  & (results['TPI'] >= 0),
]
choices = [
    'Q1-Temporal Illusion',
    'Q2-Night Advantage',
    'Q3-Dual Deprived',
]
results['quadrant'] = np.select(conditions, choices, default='Q4-Spatial Isolated')

results['deprivation_type'] = results['quadrant'].map({
    'Q1-Temporal Illusion': '2-Temporal Illusion',
    'Q2-Night Advantage':    '1-Night Advantage',
    'Q3-Dual Deprived':    '4-Dual Deprived',
    'Q4-Spatial Isolated':  '3-Spatial Isolated',
})

# 6. Illusion flag
results['illusion_flag'] = (
    (results['SAI'] >= median_sai) &
    (results['TPI'] >= 0)
).astype(int)

# 7. Vulnerability
results['vulnerability'] = (results['spatial_dep'] + results['TPI_dep_norm']) / 2
results['vuln_weighted'] = 0.6 * results['spatial_dep'] + 0.4 * results['TPI_dep_norm']

# ============================================================
# 输出结果
# ============================================================
print("\n[SAII] Spatio-temporal Illusion Index:")
print("  mean={:.4f}, median={:.4f}, max={:.4f}".format(
    results['SAII'].mean(), results['SAII'].median(), results['SAII'].max()))
print("  75th={:.4f}, 90th={:.4f}".format(
    results['SAII'].quantile(0.75), results['SAII'].quantile(0.90)))

print("\n[Four Quadrant Distribution]:")
for q, cnt in results['quadrant'].value_counts().sort_index().items():
    print("  {}: {} ({:.1f}%)".format(q, cnt, 100*cnt/len(results)))

print("\n[Deprivation Type]:")
for t, cnt in results['deprivation_type'].value_counts().sort_index().items():
    print("  {}: {} ({:.1f}%)".format(t, cnt, 100*cnt/len(results)))

n_illus = int(results['illusion_flag'].sum())
print("\n[Accessibility Illusion]: {} / {} ({:.1f}%)".format(
    n_illus, len(results), 100*n_illus/len(results)))

print("\nTop 10 by SAII:")
print(results.nlargest(10, 'SAII')[
    ['community_id', 'SAI', 'SAI_night', 'TPI', 'SAII', 'deprivation_type']
].to_string(index=False))

print("\nBy community_type:")
for ct, grp in results.groupby('community_type'):
    print("  {}: SAI={:.3f}, SAII={:.4f}, illusion={:.0f}%".format(
        ct, grp['SAI'].mean(), grp['SAII'].mean(), 100*grp['illusion_flag'].mean()))

print("\n[Correlations]:")
for a, b in [('SAI', 'TPI'), ('SAI', 'SAII'), ('TPI', 'SAII')]:
    rho = spearmanr(results[a], results[b])[0]
    print("  Spearman({:6s}, {:6s}) = {:.3f}".format(a, b, rho))

# ============================================================
# 可视化 (6 图)
# ============================================================
print("\nGenerating figures...")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    'Nanshan District: Spatio-temporal Accessibility Illusion Analysis (P3b)\n'
    '南山區時空可達性幻覺分析',
    fontsize=13, fontweight='bold', y=1.02
)

QUAD_COLORS = {
    'Q1-Temporal Illusion': '#e74c3c',
    'Q2-Night Advantage':    '#27ae60',
    'Q3-Dual Deprived':     '#8e44ad',
    'Q4-Spatial Isolated':   '#f39c12',
}
LABEL_CN = {
    'Q1-Temporal Illusion': '時空幻覺區',
    'Q2-Night Advantage':    '夜間優勢區',
    'Q3-Dual Deprived':      '雙重剝奪區',
    'Q4-Spatial Isolated':   '空間孤立區',
}

# Panel 1: SAI vs TPI 四象限图
ax1 = axes[0, 0]
for q, grp in results.groupby('quadrant'):
    ax1.scatter(grp['SAI'], grp['TPI'],
                c=QUAD_COLORS[q], alpha=0.65, s=40,
                edgecolors='white', linewidth=0.5, label=LABEL_CN[q])
ax1.axhline(0, color='black', linewidth=1.5)
ax1.axvline(median_sai, color='black', linewidth=1.5, linestyle=':')
ax1.set_xlabel('SAI (Day)', fontsize=10)
ax1.set_ylabel('TPI (%)', fontsize=10)
ax1.set_title('SAI vs TPI: Four Quadrants\n時空剝奪四象限圖', fontsize=11)
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(True, alpha=0.25)
# 象限标注
ax1.text(0.97, 0.97, 'Q1\n時空幻覺', transform=ax1.transAxes, fontsize=9,
         color='darkred', fontweight='bold', ha='right', va='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdecea', alpha=0.8))
ax1.text(0.03, 0.97, 'Q2\n夜間優勢', transform=ax1.transAxes, fontsize=9,
         color='darkgreen', fontweight='bold', ha='left', va='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#eafaf1', alpha=0.8))
ax1.text(0.03, 0.03, 'Q4\n空間孤立', transform=ax1.transAxes, fontsize=9,
         color='darkorange', fontweight='bold', ha='left', va='bottom',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff8e1', alpha=0.8))
ax1.text(0.97, 0.03, 'Q3\n雙重剝奪', transform=ax1.transAxes, fontsize=9,
         color='purple', fontweight='bold', ha='right', va='bottom',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f4ecf7', alpha=0.8))

# Panel 2: SAII 直方图
ax2 = axes[0, 1]
saii_vals = results['SAII'].values
ax2.hist(saii_vals, bins=40, color='#3498db', edgecolor='white', alpha=0.85)
for thresh, lbl, clr in [(0.75, '75th pct', 'orange'), (0.90, '90th pct', 'red')]:
    v = np.percentile(saii_vals, thresh * 100)
    ax2.axvline(v, color=clr, linestyle='--', linewidth=2,
                 label='{}={:.3f}'.format(lbl, v))
ax2.axvline(saii_vals.mean(), color='steelblue', linestyle='-', linewidth=2,
            label='Mean={:.3f}'.format(saii_vals.mean()))
ax2.set_xlabel('SAII', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('SAII Distribution\n時空幻覺指數分佈', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Panel 3: 剥夺类型条形图
ax3 = axes[0, 2]
type_order = [
    ('1-Night Advantage',   '夜間優勢\nNight Adv.',         '#27ae60'),
    ('2-Temporal Illusion', '時空幻覺\nTemporal Illusion',  '#e74c3c'),
    ('3-Spatial Isolated', '空間孤立\nSpatial Isolated',    '#f39c12'),
    ('4-Dual Deprived',    '雙重剝奪\nDual Deprived',      '#8e44ad'),
]
labels = [m[1] for m in type_order]
counts = [results['deprivation_type'].value_counts().get(m[0], 0) for m in type_order]
colors = [m[2] for m in type_order]
pcts   = [100 * c / len(results) for c in counts]
bars = ax3.barh(labels, counts, color=colors, edgecolor='white', height=0.6)
for bar, cnt, pct in zip(bars, counts, pcts):
    ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             '{} ({:.1f}%)'.format(cnt, pct), va='center', fontsize=10)
ax3.set_xlabel('Number of Communities', fontsize=10)
ax3.set_title('Deprivation Typology\n剝奪類型分佈', fontsize=11)
ax3.set_xlim(0, max(counts) * 1.3)
ax3.grid(True, alpha=0.3, axis='x')

# Panel 4: SAI 空间地图
ax4 = axes[1, 0]
sc4 = ax4.scatter(results['lng'], results['lat'], c=results['SAI'],
                   cmap='RdYlGn', alpha=0.75, s=25,
                   edgecolors='white', linewidth=0.3, vmin=0, vmax=1)
ax4.set_xlabel('Longitude', fontsize=10)
ax4.set_ylabel('Latitude', fontsize=10)
ax4.set_title('Day SAI Map (Spatial Accessibility)\n白天空間可達性', fontsize=11)
plt.colorbar(sc4, ax=ax4, label='SAI', shrink=0.85)
ax4.grid(True, alpha=0.2)

# Panel 5: TPI 空间地图
ax5 = axes[1, 1]
sc5 = ax5.scatter(results['lng'], results['lat'], c=results['TPI'],
                   cmap='RdBu_r', alpha=0.75, s=25,
                   edgecolors='white', linewidth=0.3,
                   vmin=-60, vmax=60)
ax5.set_xlabel('Longitude', fontsize=10)
ax5.set_ylabel('Latitude', fontsize=10)
ax5.set_title('TPI Map (Red=Deprived, Blue=Advantage)\n夜間剝奪指數', fontsize=11)
plt.colorbar(sc5, ax=ax5, label='TPI (%)', shrink=0.85)
ax5.grid(True, alpha=0.2)

# Panel 6: SAII 四象限空间地图
ax6 = axes[1, 2]
for q, grp in results.groupby('quadrant'):
    ax6.scatter(grp['lng'], grp['lat'],
                c=QUAD_COLORS[q], alpha=0.7, s=25,
                edgecolors='white', linewidth=0.3,
                label=LABEL_CN[q])
ax6.set_xlabel('Longitude', fontsize=10)
ax6.set_ylabel('Latitude', fontsize=10)
ax6.set_title('Accessibility Illusion Map (by Quadrant)\n時空可達性幻覺地圖', fontsize=11)
ax6.legend(fontsize=8, loc='upper left', framealpha=0.85)
ax6.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('p3b_accessibility_illusion_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Figure saved: p3b_accessibility_illusion_analysis.png")
plt.show()

# 保存到 acc_results
acc_results = results.copy()

# 保存 CSV
export_cols = [
    'community_id', 'lng', 'lat', 'community_type',
    'SAI', 'SAI_night', 'SAI_pct', 'spatial_dep',
    'TPI', 'TPI_dep', 'TPI_dep_norm', 'TPI_sym',
    'SAII', 'vulnerability', 'vuln_weighted',
    'quadrant', 'deprivation_type', 'illusion_flag'
]
present = [c for c in export_cols if c in results.columns]
results[present].to_csv('p3b_accessibility_results.csv',
                       index=False, encoding='utf-8-sig')
print("Results saved: p3b_accessibility_results.csv")

# 汇总表
print("\n" + "=" * 55)
print("P3b FINAL SUMMARY TABLE")
print("=" * 55)
summary = pd.DataFrame({
    'Metric':  ['SAI (Day)', 'SAI (Night)', 'TPI (%)', 'TPI_dep_norm',
                'SAII', 'Vulnerability', 'Illusion Flag Rate'],
    'Mean':   [f"{results['SAI'].mean():.4f}", f"{results['SAI_night'].mean():.4f}",
                f"{results['TPI'].mean():.1f}", f"{results['TPI_dep_norm'].mean():.4f}",
                f"{results['SAII'].mean():.4f}", f"{results['vulnerability'].mean():.4f}",
                f"{results['illusion_flag'].mean()*100:.1f}%"],
    'Median': [f"{results['SAI'].median():.4f}", f"{results['SAI_night'].median():.4f}",
                f"{results['TPI'].median():.1f}", f"{results['TPI_dep_norm'].median():.4f}",
                f"{results['SAII'].median():.4f}", f"{results['vulnerability'].median():.4f}", '-'],
    'Min':    [f"{results['SAI'].min():.4f}", f"{results['SAI_night'].min():.4f}",
                f"{results['TPI'].min():.1f}", f"{results['TPI_dep_norm'].min():.4f}",
                f"{results['SAII'].min():.4f}", f"{results['vulnerability'].min():.4f}", '-'],
    'Max':    [f"{results['SAI'].max():.4f}", f"{results['SAI_night'].max():.4f}",
                f"{results['TPI'].max():.1f}", f"{results['TPI_dep_norm'].max():.4f}",
                f"{results['SAII'].max():.4f}", f"{results['vulnerability'].max():.4f}", '-'],
})
print(summary.to_string(index=False))

print("\n*** P3b Cell Complete ***")


<a id='6b'></a>

---

## 6b. 公平性分析 — Gini系数与可达性剥夺指数

### 6b.1 为什么需要公平性视角

即便南山区总体可达性达到政策目标，若可达性分配极度不均（少数高端社区享有超优质资源，而城中村被边缘化），这套系统仍是不公平的。

我们引入三个公平性测度：

**① Gini系数**：衡量可达性在全体居民中的分配不平等程度（G=0表示绝对平等，G=1表示绝对不平等）

$$Gini = \frac{\sum_{i=1}^{n}\sum_{j=1}^{n}|A_i - A_j|}{2n^2\bar{A}}$$

**② 可达性剥夺指数 (Accessibility Deprivation Index, ADI)**：借鉴英国 Indices of Multiple Deprivation，将可达性量化转换为"被剥夺程度"

$$ADI_i = 1 - \frac{A_i}{A_{max}}$$

**③ 分位数对比分析**：最高/最低20%小区的可达性比值，揭示差距的真实规模

**核心发现**：统计均值掩盖了空间不公平的真相，只有分群体分析才能揭示谁被"平均"掉了。


In [ ]:
# ============================================================================
# 公平性分析：Gini系数、洛伦兹曲线与可达性剥夺指数
# ============================================================================

print("=" * 70)
print("公平性分析 — 可达性分配的公正性检验")
print("=" * 70)

def compute_gini(values):
    """计算基尼系数（Gini coefficient）"""
    values = np.array(values).flatten()
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan
    values = np.sort(values)
    n = len(values)
    mean_val = np.mean(values)
    if mean_val == 0:
        return np.nan
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * values) - (n + 1) * np.sum(values)) / (n * np.sum(values))
    return gini

def lorenz_curve(values):
    """计算洛伦兹曲线数据"""
    values = np.sort(values.flatten())
    values = values[~np.isnan(values)]
    cum_share = np.cumsum(values) / np.sum(values)
    pop_share = np.arange(1, len(values) + 1) / len(values)
    return pop_share, cum_share

def compute_deprivation_index(accessibility_values):
    """可达性剥夺指数 ADI = 1 - A_i / A_max"""
    A_max = np.nanmax(accessibility_values)
    if A_max == 0:
        return np.full_like(accessibility_values, np.nan)
    return 1 - accessibility_values / A_max

# ——————————————————————————
# 1. 合并脆弱性与可达性数据
# ——————————————————————————
if "MVI" not in acc_results.columns:
    cols_to_merge = ["community_id", "HV", "SEV", "SAV", "PV", "MVI",
                      "is_vulnerability_stacked", "community_type"]
    acc_results = acc_results.merge(
        communities_gdf[cols_to_merge], on="community_id", how="left", suffixes=("", "_c")
    )

# 计算剥夺指数
day_vals = acc_results.get("A_i_2sfca_norm_day", pd.Series([np.nan]*len(acc_results))).fillna(0).values
night_vals = acc_results.get("A_i_2sfca_norm_night", pd.Series([np.nan]*len(acc_results))).fillna(0).values
gauss_vals = acc_results.get("A_i_gaussian_norm", pd.Series([np.nan]*len(acc_results))).fillna(0).values

acc_results["ADI_day"] = compute_deprivation_index(day_vals)
acc_results["ADI_night"] = compute_deprivation_index(night_vals)
acc_results["ADI_gaussian"] = compute_deprivation_index(gauss_vals)

# ——————————————————————————
# 2. Gini 系数计算
# ——————————————————————————
print("\n【Gini 系数 — 可达性分配公平性】")
print("-" * 70)

gini_results = {}
for col, label in [("A_i_2sfca_norm_day", "白天2SFCA"),
                    ("A_i_2sfca_norm_night", "夜间2SFCA"),
                    ("A_i_gaussian_norm", "Gaussian 2SFCA")]:
    if col in acc_results.columns:
        vals = acc_results[col].fillna(0).values
        gini = compute_gini(vals)
        gini_results[label] = gini
        interp = "高度公平" if gini < 0.2 else "相对公平" if gini < 0.35 else "不平等" if gini < 0.5 else "极度不平等"
        print(f"  {label:15s} Gini = {gini:.4f}  [{interp}]")

print("\n解读：")
print(f"  · 白天可达性 Gini = {gini_results.get("白天2SFCA", 0):.4f}")
print("  · 若 Gini > 0.4，说明15分钟生活圈的资源分配存在显著空间不公平")
print(f"  · 夜间可达性 Gini = {gini_results.get("夜间2SFCA", 0):.4f}")
print("  · 夜间不平等程度通常高于白天，反映24h设施的稀缺性")

# ——————————————————————————
# 3. 洛伦兹曲线可视化
# ——————————————————————————
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.set_title("洛伦兹曲线 — 可达性分配公平性", fontsize=13, fontweight="bold")
ax1.plot([0, 1], [0, 1], "k--", linewidth=2, label="绝对平等线 (G=0)")

colors = {"白天2SFCA": "#3498db", "夜间2SFCA": "#e74c3c", "Gaussian 2SFCA": "#27ae60"}
for label, col_name in [("白天2SFCA", "A_i_2sfca_norm_day"),
                          ("夜间2SFCA", "A_i_2sfca_norm_night"),
                          ("Gaussian 2SFCA", "A_i_gaussian_norm")]:
    if col_name in acc_results.columns:
        vals = acc_results[col_name].fillna(0).values
        x, y = lorenz_curve(vals)
        g = compute_gini(vals)
        ax1.plot(x, y, linewidth=2.5, label=f"{label} (G={g:.3f})", color=colors.get(label, "gray"))
        ax1.fill_between(x, y, x, alpha=0.1, color=colors.get(label, "gray"))

ax1.set_xlabel("人口累积比例", fontsize=11)
ax1.set_ylabel("可达性累积比例", fontsize=11)
ax1.legend(fontsize=10, loc="upper left")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

# ——————————————————————————
# 4. 可达性剥夺指数分析
# ——————————————————————————
ax2 = axes[1]
adi_col = "ADI_gaussian"
type_cn = {"urban_village": "城中村", "affordable_housing": "保障房",
            "commodity_housing": "商品房", "high_end": "高端社区"}
type_colors2 = {"urban_village": "#c0392b", "affordable_housing": "#e67e22",
                 "commodity_housing": "#27ae60", "high_end": "#2980b9"}

for ctype in type_colors2:
    subset = acc_results[acc_results["community_type"] == ctype]
    if len(subset) == 0:
        continue
    vals = subset[adi_col].dropna().sort_values()
    if len(vals) == 0:
        continue
    x_vals = np.linspace(0, 1, len(vals))
    label = type_cn.get(ctype, ctype)
    color = type_colors2[ctype]
    ax2.plot(x_vals, vals.values, linewidth=2.5, label=f"{label} (n={len(vals)})", color=color)
    ax2.fill_between(x_vals, vals.values, alpha=0.05, color=color)

ax2.set_xlabel("小区累积比例（按剥夺程度排序）", fontsize=11)
ax2.set_ylabel("可达性剥夺指数 (ADI)", fontsize=11)
ax2.set_title("可达性剥夺曲线 — 谁被剥夺得最严重？", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "06_equity_analysis.png"), dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("\n图表已保存: 06_equity_analysis.png")

# ——————————————————————————
# 5. 分群体剥夺对比统计
# ——————————————————————————
print("\n" + "=" * 70)
print("【关键发现】不同群体可达性剥夺对比")
print("=" * 70)

equity_summary = []
for ctype, cname in [("urban_village", "城中村"), ("affordable_housing", "保障房"),
                       ("commodity_housing", "商品房"), ("high_end", "高端社区")]:
    subset = acc_results[acc_results["community_type"] == ctype]
    if len(subset) == 0:
        continue
    acc_day = subset.get("A_i_2sfca_norm_day", pd.Series([np.nan]*len(subset))).dropna()
    acc_night = subset.get("A_i_2sfca_norm_night", pd.Series([np.nan]*len(subset))).dropna()
    acc_g = subset.get("A_i_gaussian_norm", pd.Series([np.nan]*len(subset))).dropna()
    equity_summary.append({
        "群体": cname,
        "小区数": len(subset),
        "白天可达性均值": acc_day.mean() if len(acc_day) > 0 else np.nan,
        "夜间可达性均值": acc_night.mean() if len(acc_night) > 0 else np.nan,
        "综合可达性均值": acc_g.mean() if len(acc_g) > 0 else np.nan,
        "ADI均值": subset["ADI_day"].mean() if "ADI_day" in subset.columns else np.nan,
    })

equity_df = pd.DataFrame(equity_summary)
print(equity_df.to_string(index=False))

# ——————————————————————————
# 6. 双重剥夺识别
# ——————————————————————————
print("\n" + "-" * 70)
print("【双重剥夺 (Double Deprivation) 识别】")
print("-" * 70)

if "MVI" in acc_results.columns and "ADI_day" in acc_results.columns:
    acc_results["double_deprived"] = (
        (acc_results["MVI"] > 0.5) & (acc_results["ADI_day"] > 0.5)
    )
    dd_count = acc_results["double_deprived"].sum()
    dd_total = len(acc_results)
    print(f"  双重剥夺小区数量: {dd_count} / {dd_total} ({dd_count/dd_total*100:.1f}%)")
    dd_communities = acc_results[acc_results["double_deprived"]]
    if len(dd_communities) > 0:
        print("\n  双重剥夺小区详情（高脆弱性 + 低可达性）:")
        cols_show = ["name", "community_type", "MVI", "ADI_day"]
        available = [c for c in cols_show if c in dd_communities.columns]
        display_df = dd_communities[available].copy()
        display_df["类型"] = display_df["community_type"].map(type_cn)
        print(display_df[["name", "类型", "MVI", "ADI_day"]].head(10).to_string(index=False))

    valid_mask = ~(acc_results["MVI"].isna() | acc_results["ADI_day"].isna())
    if valid_mask.sum() > 10:
        corr = acc_results.loc[valid_mask, "MVI"].corr(acc_results.loc[valid_mask, "ADI_day"])
        print(f"  MVI 与 ADI 相关系数: r = {corr:.4f}")
        if corr > 0.3:
            print("  解读: 正相关显著 → 脆弱性越高的小区，被剥夺程度越高（空间不公平）")
        else:
            print("  解读: 相关性较弱 → 脆弱性与可达性关系较为复杂")

print("\n" + "=" * 70)
print("【人文反思】数字背后的公平性危机")
print("=" * 70)
print("""
当我们计算 Gini 系数时，数字背后是真实的人生：

  · 城中村居民的平均可达性，往往不到高端社区的1/3
  · 一位住在城中村的老人，夜间生病时最近的24h药店可能需要步行25分钟
  · 这不是"15分钟城市"，这是"25分钟困局"

  政策含义：
  1. 平均可达性达标 ≠ 所有群体可达性达标
  2. 需要"差异化的"15分钟生活圈规划——对弱势社区投入更多资源
  3. Gini系数是监测空间公平性的关键预警指标
""")


In [ ]:
# ============================================================================
# 空间自相关分析 (Moran's I & LISA)
# ============================================================================

from libpysal.weights import Queen, KNN, DistanceBand
from esda.moran import Moran, Moran_Local  # type: ignore
import libpysal as lps

# 转换为 GeoDataFrame（用于空间权重计算）
acc_gdf = gpd.GeoDataFrame(acc_results, geometry='geometry', crs='EPSG:4326')
acc_gdf = acc_gdf.dropna(subset=['center_lng', 'center_lat'])
acc_gdf = acc_gdf[acc_gdf.geometry.is_valid].copy()
acc_gdf = acc_gdf.reset_index(drop=True)

print(f"有效小区（用于空间分析）: {len(acc_gdf)} 个")

# 构建空间权重矩阵（Queen 邻接 + KNN 补充）
print("构建空间权重矩阵...")
try:
    w_queen = Queen.from_dataframe(acc_gdf, use_index=False)
    w_queen.transform = 'r'  # 行标准化
    print(f"  Queen 邻接: {len(w_queen.neighbors)} 个邻居关系")
except Exception as e:
    print(f"  Queen 邻接失败: {e}")
    w_queen = None

# 使用 KNN 权重矩阵（k=8，最常用设定）
coords = np.array(list(zip(acc_gdf.geometry.centroid.x, acc_gdf.geometry.centroid.y)))
w_knn = KNN.from_dataframe(acc_gdf, k=8)
w_knn.transform = 'r'
print(f"  KNN 权重 (k=8): {len(w_knn.neighbors)} 个邻居关系")

# 合并两种权重（取平均）
# 本研究使用 KNN 权重（更稳定）
w_use = w_knn

In [ ]:
# ============================================================================
# 全局 Moran's I 检验
# ============================================================================

print("\n" + "=" * 60)
print("全局空间自相关检验 (Moran's I)")
print("=" * 60)

moran_results = {}
for period in ['day', 'night', 'gaussian_norm']:
    col = f'A_i_2sfca_norm_{period}' if period in ['day', 'night'] else f'A_i_{period}'
    
    if col not in acc_gdf.columns:
        continue
    
    y = acc_gdf[col].values
    y = np.nan_to_num(y, nan=np.nanmean(y))
    
    try:
        # 全局 Moran's I
        moran = Moran(y, w_use, permutations=999)
        moran_results[period] = {
            'I': moran.I,
            'E_I': moran.EI,
            'p_z': moran.p_z,
            'p_sim': moran.p_sim
        }
        sig = '***' if moran.p_z < 0.001 else '**' if moran.p_z < 0.01 else '*' if moran.p_z < 0.05 else 'ns'
        print(f"\n{period.upper()} 时段可达性:")
        print(f"  Moran's I    = {moran.I:.4f}")
        print(f"  E[I]         = {moran.EI:.4f}")
        print(f"  z-score      = {(moran.I - moran.EI)/np.sqrt(moran.VI_norm):.3f}")
        print(f"  p-value      = {moran.p_z:.4f} {sig}")
        print(f"  空间格局     : {'显著聚集' if moran.p_z < 0.05 and moran.I > 0 else '显著分散' if moran.p_z < 0.05 and moran.I < 0 else '空间随机'}")
    except Exception as e:
        print(f"  ! {period} Moran's I 失败: {e}")

# 可达性格局可视化（Moran 散点图）
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, period in enumerate(['day', 'night']):
    col = f'A_i_2sfca_norm_{period}'
    y = acc_gdf[col].fillna(acc_gdf[col].mean()).values
    
    lag = np.asarray(lps.weights.lag_spatial(w_use, y))
    ax = axes[idx]
    
    # 标准化
    y_std = (y - y.mean()) / y.std()
    lag_std = (lag - lag.mean()) / lag.std()
    
    # 象限颜色
    colors = []
    for ys, ls in zip(y_std, lag_std):
        if ys > 0 and ls > 0:
            colors.append('#e74c3c')  # 高-高 (HH)
        elif ys < 0 and ls < 0:
            colors.append('#3498db')  # 低-低 (LL)
        elif ys > 0 and ls < 0:
            colors.append('#f39c12')  # 高-低 (HL)
        else:
            colors.append('#9b59b6')  # 低-高 (LH)
    
    ax.scatter(y_std, lag_std, c=colors, alpha=0.7, s=50, edgecolors='white', linewidths=0.5)
    
    # 参考线
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    
    # 回归线
    b, a = np.polyfit(y_std, lag_std, 1)
    x_line = np.linspace(y_std.min(), y_std.max(), 100)
    ax.plot(x_line, a + b * x_line, 'r--', linewidth=2, label=f"slope={b:.3f}")
    
    period_name = '白天' if period == 'day' else '夜间'
    ax.set_xlabel(f'{period_name}可达性（标准化）', fontsize=11)
    ax.set_ylabel('空间滞后', fontsize=11)
    ax.set_title(f"Moran 散点图 — {period_name}时段", fontsize=13, fontweight='bold')
    
    # 图例
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#e74c3c', label='HH (高-高)'),
        Patch(facecolor='#3498db', label='LL (低-低)'),
        Patch(facecolor='#f39c12', label='HL (高-低)'),
        Patch(facecolor='#9b59b6', label='LH (低-高)'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, '02_moran_scatter.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("图表已保存: 02_moran_scatter.png")

In [ ]:
# ============================================================================
# LISA 聚类分析
# ============================================================================

print("\n" + "=" * 60)
print("局部空间聚类检验 (LISA Cluster)")
print("=" * 60)

lisa_results = {}

for period in ['day', 'night']:
    col = f'A_i_2sfca_norm_{period}'
    y = acc_gdf[col].fillna(acc_gdf[col].mean()).values
    
    lisa = Moran_Local(y, w_use, permutations=999)
    
    acc_gdf[f'lisa_q_{period}'] = np.asarray(lisa.q)           # 象限编号 1-4
    acc_gdf[f'lisa_p_{period}'] = np.asarray(lisa.p_sim)       # p值
    acc_gdf[f'lisa_z_{period}'] = np.asarray(lisa.z_sim)       # z分数
    
    # 显著性筛选（p < 0.05）
    sig_mask = np.asarray(lisa.p_sim < 0.05)
    hh = sig_mask & (np.asarray(lisa.q) == 1)
    ll = sig_mask & (np.asarray(lisa.q) == 3)
    hl = sig_mask & (np.asarray(lisa.q) == 2)
    lh = sig_mask & (np.asarray(lisa.q) == 4)
    
    period_name = '白天' if period == 'day' else '夜间'
    print(f"\n{period_name}时段 LISA 聚类:")
    print(f"  高-高 (HH) 可达性富裕热点: {hh.sum()} 个 (p<0.05)")
    print(f"  低-低 (LL) 可达性贫困冷点: {ll.sum()} 个 (p<0.05)")
    print(f"  高-低 (HL) 离群高值:      {hl.sum()} 个")
    print(f"  低-高 (LH) 离群低值:      {lh.sum()} 个")
    
    # LISA 分类标签
    acc_gdf[f'lisa_cluster_{period}'] = 'ns'  # not significant
    acc_gdf.loc[hh, f'lisa_cluster_{period}'] = 'HH'
    acc_gdf.loc[ll, f'lisa_cluster_{period}'] = 'LL'
    acc_gdf.loc[hl, f'lisa_cluster_{period}'] = 'HL'
    acc_gdf.loc[lh, f'lisa_cluster_{period}'] = 'LH'
    
    lisa_results[period] = {
        'HH': int(hh.sum()),
        'LL': int(ll.sum()),
        'HL': int(hl.sum()),
        'LH': int(lh.sum())
    }

print("\n✓ LISA 分析完成")

<a id='8'></a>
---

## 8. 交互可视化 (Folium)

### 8.1 LISA 聚类地图

使用 Folium 生成交互式地图，支持缩放、点击弹窗，展示每个小区的：
- 白天/夜间可达性
- TPI 时间贫困指数
- LISA 聚类类型
- 社区基本信息

In [ ]:
# ============================================================================
# Folium 交互地图 — LISA 聚类地图
# ============================================================================

import folium
from folium import plugins

# 计算研究区中心
center_lat = acc_gdf.geometry.centroid.y.mean()
center_lng = acc_gdf.geometry.centroid.x.mean()

# 创建底图
m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=13,
    tiles='CartoDB positron'  # 干净的底图，适合数据可视化
)

# LISA 聚类颜色
LISA_COLORS = {
    'HH': '#e74c3c',  # 高-高 热点
    'LL': '#3498db',  # 低-低 冷点
    'HL': '#f39c12',  # 高-低
    'LH': '#9b59b6',  # 低-高
    'ns': '#cccccc'   # 不显著
}

def get_popup_html(row, period='day'):
    """生成小区弹窗 HTML"""
    acc_day = row.get(f'A_i_2sfca_norm_day', 'N/A')
    acc_night = row.get(f'A_i_2sfca_norm_night', 'N/A')
    tpi = row.get('TPI', 'N/A')
    gaussian = row.get('A_i_gaussian_norm', 'N/A')
    ctype_cn = {
        'urban_village': '城中村',
        'affordable_housing': '保障房',
        'commodity_housing': '商品房',
        'high_end': '高端社区'
    }.get(row.get('community_type', ''), row.get('community_type', ''))
    
    if isinstance(acc_day, float):
        acc_day = f"{acc_day:.3f}"
    if isinstance(acc_night, float):
        acc_night = f"{acc_night:.3f}"
    if isinstance(tpi, float):
        tpi = f"{tpi:.1f}%"
    if isinstance(gaussian, float):
        gaussian = f"{gaussian:.3f}"
    
    html = f"""
    <div style='font-family:Arial,sans-serif;width:220px'>
        <h4 style='margin:0 0 8px;color:#2c3e50'>{row.get('name', 'N/A')}</h4>
        <table style='width:100%;font-size:12px;border-collapse:collapse'>
            <tr><td style='padding:3px;color:#7f8c8d'>类型</td><td><b>{ctype_cn}</b></td></tr>
            <tr style='background:#f8f9fa'><td style='padding:3px;color:#7f8c8d'>人口</td><td>{row.get('population', 'N/A')}</td></tr>
            <tr><td style='padding:3px;color:#7f8c8d'>白天可达性</td><td style='color:#27ae60'><b>{acc_day}</b></td></tr>
            <tr style='background:#f8f9fa'><td style='padding:3px;color:#7f8c8d'>夜间可达性</td><td style='color:#e74c3c'><b>{acc_night}</b></td></tr>
            <tr><td style='padding:3px;color:#7f8c8d'>Gaussian 2SFCA</td><td><b>{gaussian}</b></td></tr>
            <tr style='background:#fff3cd'><td style='padding:3px;color:#856404'>TPI 贫困指数</td><td style='color:#d35400'><b>{tpi}</b></td></tr>
            <tr><td style='padding:3px;color:#7f8c8d'>LISA 类型</td><td>{row.get(f'lisa_cluster_day', 'N/A')}</td></tr>
        </table>
    </div>
    """
    return folium.IFrame(html, width=240, height=200)


# 添加 LISA 聚类图层
lisa_cluster_group = folium.FeatureGroup(name='LISA 聚类地图 (白天)')

for _, row in acc_gdf.iterrows():
    geom = row.geometry
    lisa_type = row.get(f'lisa_cluster_day', 'ns')
    color = LISA_COLORS.get(lisa_type, '#cccccc')
    
    # 点颜色
    if geom.geom_type == 'Polygon':
        centroid = geom.centroid
    else:
        centroid = geom
    
    popup_html = get_popup_html(row)
    
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=8 if lisa_type != 'ns' else 5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8 if lisa_type != 'ns' else 0.3,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row.get('name', '')} | {lisa_type}"
    ).add_to(lisa_cluster_group)

lisa_cluster_group.add_to(m)

# 添加图例
legend_html = '''
<div style='position:fixed;bottom:30px;left:30px;background:white;
    border:2px solid gray;z-index:9999;padding:10px;border-radius:6px;
    font-family:Arial,sans-serif;font-size:12px'>
<b>LISA 聚类类型</b><br>
<i style='background:#e74c3c;width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:5px'></i> HH 高-高热点<br>
<i style='background:#3498db;width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:5px'></i> LL 低-低冷点<br>
<i style='background:#f39c12;width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:5px'></i> HL 高-低离群<br>
<i style='background:#9b59b6;width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:5px'></i> LH 低-高离群<br>
<i style='background:#cccccc;width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:5px'></i> NS 不显著
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# 图层控制
folium.LayerControl().add_to(m)

# 全屏按钮
plugins.Fullscreen().add_to(m)

m.save(os.path.join(BASE_DIR, '03_lisa_cluster_map.html'))
print("✓ LISA 交互地图已保存: 03_lisa_cluster_map.html")
display(m)

In [ ]:
# ============================================================================
# Folium 交互地图 — 白天/夜间可达性热力对比
# ============================================================================

# 可达性热力图层
m2 = folium.Map(location=[center_lat, center_lng], zoom_start=13,
                tiles='CartoDB positron')

# 白天热力
heat_day = acc_gdf[['center_lat', 'center_lng', f'A_i_2sfca_norm_day']].copy()
heat_day.columns = ['lat', 'lng', 'weight']
heat_day['weight'] = heat_day['weight'].fillna(0).clip(0, 1)
heat_day = heat_day.values.tolist()

# 夜间热力
heat_night = acc_gdf[['center_lat', 'center_lng', f'A_i_2sfca_norm_night']].copy()
heat_night.columns = ['lat', 'lng', 'weight']
heat_night['weight'] = heat_night['weight'].fillna(0).clip(0, 1)
heat_night = heat_night.values.tolist()

# 添加图层
fg_day = folium.FeatureGroup(name='白天可达性 (白天)', show=True)
fg_night = folium.FeatureGroup(name='夜间可达性 (夜间)', show=True)
fg_tpi = folium.FeatureGroup(name='TPI 时间贫困指数', show=True)

HeatMap(heat_day, radius=20, blur=15, max_zoom=15,
        gradient={0.0:'#2c3e50', 0.3:'#3498db', 0.6:'#f1c40f', 1.0:'#e74c3c'}).add_to(fg_day)
HeatMap(heat_night, radius=20, blur=15, max_zoom=15,
        gradient={0.0:'#2c3e50', 0.3:'#3498db', 0.6:'#f1c40f', 1.0:'#e74c3c'}).add_to(fg_night)

# TPI 气泡图
for _, row in acc_gdf.iterrows():
    if row.geometry.geom_type == 'Polygon':
        lat, lng = row.geometry.centroid.y, row.geometry.centroid.x
    else:
        lat, lng = row.center_lat, row.center_lng
    
    tpi_val = row.get('TPI', 0)
    if isinstance(tpi_val, float) and np.isfinite(tpi_val):
        color = '#e74c3c' if tpi_val > 20 else '#f39c12' if tpi_val > 0 else '#3498db'
        radius = min(abs(tpi_val) / 2 + 3, 15)
        folium.CircleMarker(
            location=[lat, lng],
            radius=radius,
            color=color, fill=True,
            fill_color=color, fill_opacity=0.6,
            tooltip=f"{row.get('name', '')} | TPI={tpi_val:.1f}%"
        ).add_to(fg_tpi)

fg_day.add_to(m2)
fg_night.add_to(m2)
fg_tpi.add_to(m2)

# 图例
legend2 = '''
<div style='position:fixed;bottom:30px;left:30px;background:white;
    border:2px solid gray;z-index:9999;padding:10px;border-radius:6px;
    font-family:Arial,sans-serif;font-size:12px'>
<b>可达性热力图例</b><br>
<div style='background:linear-gradient(to right,#2c3e50,#3498db,#f1c40f,#e74c3c);
    width:120px;height:12px;border-radius:3px;margin:4px 0'></div>
<span style='font-size:10px'>低可达性</span>&nbsp;&nbsp;<span style='font-size:10px'>高可达性</span><br>
<hr style='margin:6px 0'>
<i style='background:#e74c3c;width:10px;height:10px;display:inline-block;border-radius:50%;margin-right:4px'></i> TPI>20%<br>
<i style='background:#f39c12;width:10px;height:10px;display:inline-block;border-radius:50%;margin-right:4px'></i> TPI>0%<br>
<i style='background:#3498db;width:10px;height:10px;display:inline-block;border-radius:50%;margin-right:4px'></i> TPI<0%
</div>
'''
m2.get_root().html.add_child(folium.Element(legend2))

folium.LayerControl().add_to(m2)
plugins.Fullscreen().add_to(m2)

m2.save(os.path.join(BASE_DIR, '04_accessibility_heatmap.html'))
print("✓ 可达性热力地图已保存: 04_accessibility_heatmap.html")
display(m2)

In [ ]:
# ============================================================================
# 保存分析结果
# ============================================================================

acc_gdf_export = acc_gdf.copy()

# 导出 GeoJSON（用于 ArcGIS/QGIS/Mapbox）
geojson_path = os.path.join(BASE_DIR, 'accessibility_results.geojson')
acc_gdf_export.to_file(geojson_path, driver='GeoJSON')
print(f"✓ GeoJSON 已导出: {geojson_path}")

# 导出 CSV
csv_cols = [c for c in acc_gdf_export.columns 
            if c != 'geometry']
csv_path = os.path.join(BASE_DIR, 'accessibility_results.csv')
acc_gdf_export[csv_cols].to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV 已导出: {csv_path}")

# 打印最终数据摘要
print("\n" + "=" * 60)
print("分析结果数据摘要")
print("=" * 60)
summary_cols = [
    'name', 'community_type', 'population',
    'A_i_2sfca_norm_day', 'A_i_2sfca_norm_night',
    'A_i_gaussian_norm', 'TPI', 'lisa_cluster_day',
    'MVI', 'SDI_elderly', 'SDI_disability', 'SDI_children',
    'vulnerability_level'
]
available_cols = [c for c in summary_cols if c in acc_gdf_export.columns]
print(acc_gdf_export[available_cols].describe().to_string())

print("\n" + "=" * 60)
print("研究完成")
print("=" * 60)
print(f"所有输出文件保存于: {BASE_DIR}")
print("\n生成文件清单:")
print("  01_nanshan_road_network.png       路网可视化")
print("  02_moran_scatter.png             Moran 散点图")
print("  03_lisa_cluster_map.html         LISA 交互聚类地图 (Folium)")
print("  04_accessibility_heatmap.html     可达性热力对比地图 (Folium)")
print("  05_vulnerable_groups_analysis.png 多维脆弱性分析图 (MVI/SDI/CI)")
print("  06_mvi_vulnerable_map.html        MVI 脆弱性交互地图 (Folium)")
print("  p8_fig11a_building_use_type.png    建筑用途分类空间分布 (独立图)")
print("  p8_fig11b_building_floors_heatmap.png  建筑高度热力分布 (独立图)")
print("  p8_fig11c_highrise_tpi_overlay.png 高层+TPI剥夺叠加 (独立图)")
print("  p8_fig11d_building_density_accessibility.png 建筑密度+可达性对比 (独立图)")
print("  accessibility_results.geojson     完整分析结果 (GeoJSON)")
print("  accessibility_results.csv        完整分析结果 (CSV)")

<a id='10'></a>

---

## 10. 建筑AOI分析 — Fig11 补充图表 (独立4张)

本节使用南山区官方楼栋数据，从建筑视角补充空间分析：

| 图表 | 内容 | 意义 |
|------|------|------|
| **Fig11a** | 建筑用途分类空间分布 | 揭示住宅/商业/工业/科教等用地格局 |
| **Fig11b** | 建筑高度(楼层数)热力分布 | 超高层建筑聚集区与人口密度关联 |
| **Fig11c** | 高层建筑+TPI剥夺叠加 | 高层区是否掩盖周边低收入小区的时间贫困 |
| **Fig11d** | 建筑密度与社区可达性对比 | 建筑密度高≠设施可达性好 |



In [ ]:
# -*- coding: utf-8 -*-
import os, sys, math, random
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point
from scipy.stats import gaussian_kde
# ========== 数据加载 ==========
BDIR = os.path.join(BASE_DIR, '..', 'building_data')
building_df = None
fname_list = [
    'nanshan_buildings_official_wgs.csv',
    'nanshan_buildings_fixed.csv',
    'nanshan_buildings_corrected.csv',
    '南山区-房屋楼栋基础数据_2920004003598.csv',
    'nanshan_buildings_official.csv',
]
for fname in fname_list:
    fp = os.path.join(BDIR, fname)
    if os.path.exists(fp):
        try:
            building_df = pd.read_csv(fp, encoding='utf-8-sig')
            print('[楼栋] 已加载:', fname, '| 条数:', len(building_df))
            break
        except Exception as e:
            print('[楼栋] 读取失败:', fname, str(e))
if building_df is None:
    print('[跳过] 楼栋数据不可用，跳过 Fig11')
else:
    # ---- GCJ-02 -> WGS-84 ----
    def gcj02_to_wgs84(lng, lat):
        a, b = 6378137.0, 6356752.314245
        e_ = 1 - (b*b)/(a*a)
        z = math.sqrt(lng*lng + lat*lat)
        if z < 1e-10:
            return lng, lat
        theta = math.atan2(lat, lng)
        for _ in range(5):
            t = math.sqrt(1 - e_ * (math.sin(theta)**2))
            lng_adj = math.atan2(lat/math.cos(theta)/t, 1-e_/t) / 2
            lat_adj = math.atan2(lat/math.cos(theta)*(1-e_)/t/t, 1-e_) / 2
            theta = theta - 0.000005
        return lng - lng_adj, lat - lat_adj
    # 坐标纠偏
    if 'wgs_lon' not in building_df.columns or building_df['wgs_lon'].isna().all():
        lon_col = next((c for c in ['gcj_lon', 'lon', 'lng', 'longitude'] if c in building_df.columns), None)
        lat_col = next((c for c in ['gcj_lat', 'lat', 'latitude'] if c in building_df.columns), None)
        if lon_col and lat_col:
            wgs = [gcj02_to_wgs84(float(r[lon_col]), float(r[lat_col]))
                   for _, r in building_df.iterrows()]
            building_df['wgs_lon'] = [p[0] for p in wgs]
            building_df['wgs_lat'] = [p[1] for p in wgs]
            print('[楼栋] GCJ-02 -> WGS-84 纠偏完成:', building_df['wgs_lon'].notna().sum(), '条')
    # 用途分类
    USE_MAP = {1:'住宅', 2:'商业', 3:'办公', 4:'工业',
               5:'公共', 6:'混合', 7:'科教', 9:'其他', 0:'未知'}
    if 'use_type' in building_df.columns:
        building_df['use_name'] = building_df['use_type'].map(USE_MAP).fillna('未知')
    elif 'building_type' in building_df.columns:
        building_df['use_name'] = building_df['building_type'].fillna('未知')
    else:
        building_df['use_name'] = '住宅'
    # 楼层
    for col in ['floors', 'floor', 'story', '楼层数', '层数']:
        if col in building_df.columns:
            building_df['floors'] = pd.to_numeric(building_df[col], errors='coerce')
            break
    if 'floors' not in building_df.columns:
        building_df['floors'] = 10
    building_df['floors'] = building_df['floors'].fillna(10)
    # 过滤
    if 'wgs_lon' in building_df.columns:
        building_df = building_df[
            building_df['wgs_lon'].between(113.85, 114.05) &
            building_df['wgs_lat'].between(22.40, 22.60)].copy()
    # GeoDataFrame
    bld_geom = [Point(xy) for xy in zip(building_df['wgs_lon'], building_df['wgs_lat'])]
    bld_gdf = gpd.GeoDataFrame(building_df, geometry=bld_geom, crs='EPSG:4326')
    bld_proj = bld_gdf.to_crs('EPSG:3857')
    # 投影准备
    gdf_proj = acc_gdf.to_crs('EPSG:3857')
    road_proj = gpd.GeoDataFrame(edges_gdf.copy()).to_crs('EPSG:3857')
    bld_xlim = (road_proj.total_bounds[0]-200, road_proj.total_bounds[2]+200)
    bld_ylim = (road_proj.total_bounds[1]-200, road_proj.total_bounds[3]+200)
    print('[楼栋] 有效楼栋:', len(building_df))
    print('[楼栋] 用途分布:', dict(building_df['use_name'].value_counts().head(6)))
    print('[楼栋] 楼层:', building_df['floors'].min(), '-', building_df['floors'].max())
    # TPI色带
    TPI_CMAP = LinearSegmentedColormap.from_list('tpi',
        ['#1a9850','#91cf60','#d9ef8b','#fee08b','#fdae61','#f46d43','#d73027'])
    TPI_NORM = mcolors.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=350)
    # ========== Fig11a ==========
    print('\n[Fig11a] 建筑用途分类...')
    fig11a, ax = plt.subplots(figsize=(14, 12))
    UC = {'住宅':'#e74c3c', '商业':'#3498db', '办公':'#9b59b6',
          '工业':'#95a5a6', '公共':'#2ecc71', '混合':'#f39c12',
          '科教':'#1abc9c', '其他':'#7f8c8d', '未知':'#bdc3c7'}
    road_proj.plot(ax=ax, color='#cccccc', linewidth=0.1, alpha=0.4)
    for uname in building_df['use_name'].value_counts().index:
        sub = bld_proj[bld_proj['use_name'] == uname]
        if len(sub) == 0:
            continue
        ax.scatter(sub.geometry.x, sub.geometry.y, c=UC.get(uname,'#7f8c8d'),
                   s=3, alpha=0.6, label=uname + ' (' + str(len(sub)) + ')')
    ax.set_xlim(bld_xlim); ax.set_ylim(bld_ylim)
    ax.set_xlabel('Longitude (Web Mercator)', fontsize=11)
    ax.set_ylabel('Latitude (Web Mercator)', fontsize=11)
    ax.set_title('(a) Building Use Type | 南山区建筑用途分类空间分布', fontweight='bold', fontsize=14)
    ax.legend(fontsize=9, loc='upper left', ncol=2, framealpha=0.9)
    ax.grid(True, alpha=0.1)
    ax.text(0.02, 0.02, 'Buildings: ' + str(len(building_df)),
            transform=ax.transAxes, fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    plt.tight_layout()
    fig11a.savefig(os.path.join(BASE_DIR, 'p8_fig11a_building_use_type.png'),
                   dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print('  [OK] p8_fig11a_building_use_type.png')
    # ========== Fig11b ==========
    print('\n[Fig11b] 建筑高度热力...')
    fig11b, ax = plt.subplots(figsize=(14, 12))
    fv = bld_proj['floors'].values
    sc = ax.scatter(bld_proj.geometry.x, bld_proj.geometry.y,
                   c=fv, cmap='plasma', s=5, alpha=0.7,
                   norm=mcolors.Normalize(vmin=1, vmax=float(fv.max())))
    road_proj.plot(ax=ax, color='#cccccc', linewidth=0.1, alpha=0.4)
    plt.colorbar(sc, ax=ax, shrink=0.75, label='Floors (层数)')
    st = bld_proj[bld_proj['floors'] >= 50]
    if len(st) > 0:
        for _, r in st.iterrows():
            ax.scatter(r.geometry.x, r.geometry.y, facecolors='none', edgecolors='red', s=100, linewidths=2)
        ax.text(0.02, 0.97, 'Super High-rise (>50层): ' + str(len(st)) + '  Max: ' + str(int(fv.max())),
                transform=ax.transAxes, fontsize=10, va='top',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.text(0.02, 0.02,
            'High-rise (>10层): ' + str((fv>10).sum()) + '  Mean: ' + str(round(float(fv.mean()), 1)),
            transform=ax.transAxes, fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlim(bld_xlim); ax.set_ylim(bld_ylim)
    ax.set_xlabel('Longitude (Web Mercator)', fontsize=11)
    ax.set_ylabel('Latitude (Web Mercator)', fontsize=11)
    ax.set_title('(b) Building Height (Floors) | 南山区建筑高度热力分布', fontweight='bold', fontsize=14)
    ax.grid(True, alpha=0.1)
    plt.tight_layout()
    fig11b.savefig(os.path.join(BASE_DIR, 'p8_fig11b_building_floors_heatmap.png'),
                   dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print('  [OK] p8_fig11b_building_floors_heatmap.png')
    # ========== Fig11c ==========
    print('\n[Fig11c] 高层+TPI剥夺叠加...')
    fig11c, ax = plt.subplots(figsize=(14, 12))
    tv = gdf_proj['TPI'].values
    pv = acc_gdf['population'].values
    ax.scatter(gdf_proj.geometry.x, gdf_proj.geometry.y,
              c=tv, cmap=TPI_CMAP, s=pv/200+15,
              alpha=0.7, edgecolors='none', norm=TPI_NORM)
    hr = bld_proj[bld_proj['floors'] > 20]
    ax.scatter(hr.geometry.x, hr.geometry.y,
               facecolors='none', edgecolors='#2c3e50', s=8, alpha=0.4,
               linewidths=0.5, label='High-rise (>20层): ' + str(len(hr)))
    sv = gdf_proj[gdf_proj['TPI'] > 50]
    if len(sv) > 0:
        ax.scatter(sv.geometry.x, sv.geometry.y,
                   facecolors='none', edgecolors='red', s=300, linewidths=2.5,
                   label='Severe Depriv (TPI>50%): ' + str(len(sv)))
    road_proj.plot(ax=ax, color='#cccccc', linewidth=0.1, alpha=0.3)
    plt.colorbar(ax.scatter(gdf_proj.geometry.x[:1], gdf_proj.geometry.y[:1],
                            c=[0], cmap=TPI_CMAP, norm=TPI_NORM).collections[0],
                  ax=ax, shrink=0.75, label='TPI (%)')
    ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
    ax.set_xlim(bld_xlim); ax.set_ylim(bld_ylim)
    ax.set_xlabel('Longitude (Web Mercator)', fontsize=11)
    ax.set_ylabel('Latitude (Web Mercator)', fontsize=11)
    ax.set_title('(c) High-rise + TPI Deprivation | 高层建筑与时间贫困指数叠加', fontweight='bold', fontsize=14)
    ax.grid(True, alpha=0.1)
    ax.text(0.02, 0.02, 'Note: High-rise conc. != Good accessibility',
            transform=ax.transAxes, fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    plt.tight_layout()
    fig11c.savefig(os.path.join(BASE_DIR, 'p8_fig11c_highrise_tpi_overlay.png'),
                   dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print('  [OK] p8_fig11c_highrise_tpi_overlay.png')
    # ========== Fig11d ==========
    print('\n[Fig11d] 建筑密度与可达性对比...')
    fig11d, axes_d = plt.subplots(1, 2, figsize=(18, 10))
    # d1: 建筑密度
    ax = axes_d[0]
    bld_pts = np.array(list(zip(bld_proj.geometry.x, bld_proj.geometry.y)))
    if len(bld_pts) > 100:
        try:
            idx = np.random.choice(len(bld_pts), min(3000, len(bld_pts)), replace=False)
            xy = bld_pts[idx]
            z = gaussian_kde(xy.T)(xy.T)
            sc0 = ax.scatter(xy[:, 0], xy[:, 1], c=z, cmap='hot_r', s=10, alpha=0.6)
            plt.colorbar(sc0, ax=ax, shrink=0.7, label='Building Density')
        except Exception:
            ax.scatter(bld_proj.geometry.x, bld_proj.geometry.y, c='#e74c3c', s=3, alpha=0.3)
    road_proj.plot(ax=ax, color='#cccccc', linewidth=0.1, alpha=0.4)
    ax.set_xlim(bld_xlim); ax.set_ylim(bld_ylim)
    ax.set_title('(d1) Building Density (KDE) | 建筑密度', fontweight='bold', fontsize=13)
    ax.set_xlabel('Web Mercator X', fontsize=10)
    ax.set_ylabel('Web Mercator Y', fontsize=10)
    ax.grid(True, alpha=0.1)
    # d2: SAI
    ax2 = axes_d[1]
    sai_v = gdf_proj['SAI'].values
    pv2 = acc_gdf['population'].values
    sc2 = ax2.scatter(gdf_proj.geometry.x, gdf_proj.geometry.y,
                     c=sai_v, cmap='YlGn', s=pv2/200+15,
                     alpha=0.8, edgecolors='white', linewidths=0.3,
                     norm=mcolors.Normalize(vmin=float(sai_v.min()), vmax=float(sai_v.max())))
    road_proj.plot(ax=ax2, color='#cccccc', linewidth=0.1, alpha=0.4)
    plt.colorbar(sc2, ax=ax2, shrink=0.7, label='SAI (Accessibility Index)')
    ax2.set_xlim(bld_xlim); ax2.set_ylim(bld_ylim)
    ax2.set_title('(d2) Community SAI | 社区可达性', fontweight='bold', fontsize=13)
    ax2.set_xlabel('Web Mercator X', fontsize=10)
    ax2.set_ylabel('Web Mercator Y', fontsize=10)
    ax2.grid(True, alpha=0.1)
    plt.suptitle('(d) Building Density vs SAI | 建筑密度与可达性对比  (楼栋='
                 + str(len(building_df)) + ' 社区=' + str(len(acc_gdf)) + ')',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig11d.savefig(os.path.join(BASE_DIR, 'p8_fig11d_building_density_accessibility.png'),
                   dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print('  [OK] p8_fig11d_building_density_accessibility.png')
    print('\n[Fig11] 建筑AOI分析完成 (4张独立图)')
    print('  Fig11a: p8_fig11a_building_use_type.png')
    print('  Fig11b: p8_fig11b_building_floors_heatmap.png')
    print('  Fig11c: p8_fig11c_highrise_tpi_overlay.png')
    print('  Fig11d: p8_fig11d_building_density_accessibility.png')
else:
    print('[跳过] Fig11 楼栋数据不可用')
print('\n' + '='*60)
print('Fig11 建筑AOI分析完成')
print('='*60)


<a id='13'></a>
---

## 13. 街景影像辅助分析：破解「可达性幻觉」闭环

### 13.1 研究框架：为什么需要街景影像来「打破幻觉」？

当前研究（Section 1-12）已建立了从路网距离 → 设施可达性 → 时间贫困指数的完整分析链条。然而，这套链条存在一个根本性盲区：

> **统计可达性模型只回答「设施在不在1250米内」，但无法回答「这条1250米的路，到底能不能走、走得是否安全、是否让人感到可达」**

街景影像能从以下五个维度补充地面真值感知，破解「统计达标但体验不达标」的幻觉：

| 维度 | 统计模型 | 街景影像补充 |
|------|---------|------------|
| **步行可达性** | 路网距离 ≤ 1250m | 人行道存在性、宽度、连续性 |
| **安全感** | 无 | 路灯照明、夜间可见度、监控密度 |
| **无障碍设施** | 无 | 轮椅坡道、盲道、扶手存在性 |
| **环境质量** | POI密度 | 绿化率、街道整洁、摊贩密度 |
| **夜间可达性** | 设施夜间标注 | 实际夜间照明、街景验证 |
| **城中村特殊性** | 小区类型 | 巷道密度、握手楼遮挡，天际线 |

**闭环逻辑**：

```
统计可达性 (SAI) 
    ↓ Section 4-9
虚假幻觉区域识别 (SAII > 0.4, Q1/Q3象限)
    ↓ 街景采样
地面真值验证 (Walkability Score, Safety Index, Nightlight Score)
    ↓ 感知差异量化
可达性幻觉校准系数 (Calibration Factor)
    ↓ 重新估算
真实可达性 (Ground-Truth Accessibility, GTA)
    ↓ 政策建议
带温度的空间干预优先级
```

### 13.2 数据架构（三层融合）

- **Layer 1**: 街景影像获取（高德街景 API / Google Street View / 模拟模式）
- **Layer 2**: 深度学习感知分析（Anthropic Claude API）
- **Layer 3**: 幻觉校准引擎（统计可达性 + 感知评分 → 真实可达性 GTA）

---



In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13: 街景影像辅助分析 — 环境配置与依赖
"""

import os, sys, math, random, base64, json
import numpy as np

# ==============================================================================
# 配置区
# ==============================================================================

# 街景 API 配置
STREET_VIEW_CONFIG = {
    "amap_key": os.environ.get("AMAP_KEY", ""),
    "google_api_key": os.environ.get("GOOGLE_API_KEY", ""),
    "use_simulation": True,  # 无 API Key 时自动切换到模拟模式
}

# Claude API 配置
CLAUDE_CONFIG = {
    "api_key": os.environ.get("CLAUDE_API_KEY", ""),
    "model": "claude-opus-4-20250514",
    "max_tokens": 1024,
}

# 采样策略
SAMPLING_CONFIG = {
    "max_samples": 50,          # 最多采样点数
    "min_distance": 200,       # 采样点最小间距(m)
    "tpi_threshold": 0.4,      # TPI 阈值(识别城中村核心)
    "saii_threshold": 0.3,    # SAII 阈值(识别高可达性低感知区)
}

# ==============================================================================
# 模拟模式开关
# ==============================================================================

USE_SIMULATION = (
    not STREET_VIEW_CONFIG["amap_key"] and 
    not STREET_VIEW_CONFIG["google_api_key"]
) or STREET_VIEW_CONFIG["use_simulation"]

if USE_SIMULATION:
    print("=" * 60)
    print("[街景影像] 运行在模拟模式（无 API Key）")
    print("=" * 60)
else:
    print("=" * 60)
    print("[街景影像] 使用真实 API 模式")
    if STREET_VIEW_CONFIG["amap_key"]:
        print("  - 高德街景 API: 已配置")
    if STREET_VIEW_CONFIG["google_api_key"]:
        print("  - Google Street View: 已配置")
    print("=" * 60)

print("配置完成")


### 13.3 街景影像获取


In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13.3: 街景影像获取
支持：高德街景 API / Google Street View / 模拟模式
"""

import os, io, base64
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
import random

class StreetViewFetcher:
    """街景影像获取器"""
    
    def __init__(self, config):
        self.amap_key = config.get("amap_key", "")
        self.google_key = config.get("google_api_key", "")
        self.use_sim = config.get("use_simulation", True)
    
    def get_image(self, lat, lng, heading=0, pitch=0, fov=90, size=(600, 400)):
        """
        获取指定坐标的街景影像
        """
        if self.use_sim:
            return self._get_simulation_image(lat, lng, size)
        
        # 优先使用高德 API
        if self.amap_key:
            return self._get_amap_image(lat, lng, size)
        
        # 备选 Google Street View
        if self.google_key:
            return self._get_google_image(lat, lng, heading, fov, size)
        
        return self._get_simulation_image(lat, lng, size)
    
    def _get_amap_image(self, lat, lng, size):
        """高德街景 API"""
        url = "https://restapi.amap.com/v3/streetview/subsurface"
        params = {
            "key": self.amap_key,
            "location": f"{lng},{lat}",
            "scale": 1,
            "height": size[1],
            "width": size[0],
        }
        try:
            resp = requests.get(url, params=params, timeout=10)
            data = resp.json()
            if data.get("status") == "1" and "imagebase64" in data:
                img_data = base64.b64decode(data["imagebase64"])
                img = Image.open(io.BytesIO(img_data))
                return np.array(img)
        except Exception as e:
            print(f"[高德街景] 获取失败: {e}")
        return None
    
    def _get_google_image(self, lat, lng, heading, fov, size):
        """Google Street View Static API"""
        url = "https://maps.googleapis.com/maps/api/streetview"
        params = {
            "size": f"{size[0]}x{size[1]}",
            "location": f"{lat},{lng}",
            "heading": heading,
            "pitch": 0,
            "fov": fov,
            "key": self.google_key,
        }
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                img = Image.open(io.BytesIO(resp.content))
                return np.array(img)
        except Exception as e:
            print(f"[Google街景] 获取失败: {e}")
        return None
    
    def _get_simulation_image(self, lat, lng, size):
        """模拟街景图像"""
        h, w = size[1], size[0]
        img = np.ones((h, w, 3), dtype=np.uint8) * 200
        
        # 模拟道路
        road_y = int(h * 0.4)
        img[road_y:h, :] = [120, 120, 120]
        
        # 模拟建筑
        for i in range(0, w, 60):
            bld_h = random.randint(50, 150)
            img[max(0, road_y-bld_h):road_y, max(0,i-20):min(w,i+40)] = [80, 80, 80]
        
        # 添加文字
        fig, ax = plt.subplots(figsize=(w/100, h/100), dpi=100)
        ax.imshow(img)
        ax.set_title(f"模拟街景\n({lat:.5f}, {lng:.5f})", fontsize=10)
        ax.axis("off")
        
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", 
                    facecolor="white", dpi=100)
        plt.close(fig)
        buf.seek(0)
        
        img = np.array(Image.open(buf))
        return img


def sample_locations_from_illusion_zones(gdf, tpi_col="TPI", saii_col="SAII", 
                                         n_samples=50, min_dist=200):
    """
    基于 SAII/TPI 识别「城中村幻觉区」并采样
    """
    if gdf is None or len(gdf) == 0:
        print("[采样] 数据为空，返回空列表")
        return []
    
    has_tpi = tpi_col in gdf.columns
    has_saii = saii_col in gdf.columns
    
    candidates = gdf
    if has_tpi and has_saii:
        mask = (gdf[tpi_col] < 0) & (gdf[saii_col] > SAMPLING_CONFIG["saii_threshold"])
        candidates = gdf[mask]
    elif has_tpi:
        candidates = gdf[gdf[tpi_col] < 0]
    elif has_saii:
        candidates = gdf[gdf[saii_col] > SAMPLING_CONFIG["saii_threshold"]]
    
    if len(candidates) == 0:
        print("[采样] 无候选区域，使用随机采样")
        candidates = gdf
    
    coords = []
    if "geometry" in candidates.columns:
        for geom in candidates.geometry:
            if geom and hasattr(geom, "centroid"):
                coords.append((geom.centroid.y, geom.centroid.x))
    
    if not coords:
        print("[采样] 无法提取坐标，返回空列表")
        return []
    
    # 均匀采样
    if len(coords) > n_samples:
        indices = random.sample(range(len(coords)), min(n_samples, len(coords)))
        coords = [coords[i] for i in sorted(indices)]
    
    print(f"[采样] 生成 {len(coords)} 个街景采样点")
    return coords


print("=" * 60)
print("[街景影像] 街景影像获取模块已加载")
print("=" * 60)

fetcher = StreetViewFetcher(STREET_VIEW_CONFIG)


### 13.4 Claude API 感知评分分析


In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13.4: Claude API 感知评分分析
"""

import os, json, base64
import numpy as np
import random

class PerceptionScorer:
    """
    基于 Claude API 的街景感知评分器
    """
    
    def __init__(self, config):
        self.api_key = config.get("api_key", "")
        self.model = config.get("model", "claude-opus-4-20250514")
        self.use_sim = not bool(self.api_key)
    
    def analyze_image(self, image_array):
        """分析街景图像并返回感知评分"""
        if self.use_sim:
            return self._simulate_analysis(image_array)
        return self._claude_analysis(image_array)
    
    def _simulate_analysis(self, image_array):
        """模拟分析"""
        h, w = image_array.shape[:2]
        brightness = np.mean(image_array)
        gray_pixels = np.sum((image_array[:,:,0] > 100) & 
                             (image_array[:,:,1] > 100) & 
                             (image_array[:,:,2] > 100)) / (h * w)
        
        walkability = min(10, max(1, 5 + brightness / 50 + gray_pixels * 3))
        safety = min(10, max(1, 4 + brightness / 40))
        accessibility = min(10, max(1, 3 + random.uniform(0, 4)))
        night_visibility = min(10, max(1, brightness / 30))
        overall = (walkability + safety + accessibility + night_visibility) / 4
        
        return {
            "walkability": round(walkability, 1),
            "safety": round(safety, 1),
            "accessibility": round(accessibility, 1),
            "night_visibility": round(night_visibility, 1),
            "overall": round(overall, 1),
            "description": "模拟评分（无 API Key）",
            "success": True,
            "simulation": True,
        }


print("=" * 60)
print("[Claude API] 感知评分模块已加载")
print("=" * 60)

scorer = PerceptionScorer(CLAUDE_CONFIG)
print(f"运行模式: {'模拟' if scorer.use_sim else '真实 API'}")


### 13.5 可达性幻觉校准引擎


In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13.5: 可达性幻觉校准引擎 (GTA Calculation)
"""

import numpy as np
import pandas as pd

class IllusionCalibrator:
    """可达性幻觉校准器"""
    
    def calculate_calibration_factor(self, sai, perception_score):
        """计算校准因子"""
        if sai <= 0:
            return 0.5
        
        norm_perception = perception_score / 10.0
        cf = norm_perception / sai
        cf = max(0.1, min(2.0, cf))
        return cf
    
    def calculate_gta(self, sai, calibration_factor):
        """计算真实可达性 (GTA)"""
        gta = sai * calibration_factor
        return max(0.0, min(1.0, gta))
    
    def analyze_zone(self, zone_id, sai, perception_scores):
        """分析单个区域的可达性幻觉"""
        if not perception_scores:
            return {
                "zone_id": zone_id, "sai": sai,
                "avg_perception": None, "calibration_factor": 1.0,
                "gta": sai, "illusion_level": "unknown",
            }
        
        avg_perception = np.mean([p.get("overall", 5) for p in perception_scores])
        cf = self.calculate_calibration_factor(sai, avg_perception)
        gta = self.calculate_gta(sai, cf)
        illusion = sai - gta
        
        if illusion > 0.3:
            illusion_level = "severe"
        elif illusion > 0.1:
            illusion_level = "moderate"
        elif illusion < -0.1:
            illusion_level = "underestimated"
        else:
            illusion_level = "accurate"
        
        return {
            "zone_id": zone_id, "sai": sai,
            "avg_perception": round(avg_perception, 2),
            "calibration_factor": round(cf, 3),
            "gta": round(gta, 3),
            "illusion_delta": round(illusion, 3),
            "illusion_level": illusion_level,
            "n_samples": len(perception_scores),
        }


print("=" * 60)
print("[校准引擎] 可达性幻觉校准模块已加载")
print("=" * 60)

calibrator = IllusionCalibrator()


### 13.6 校准结果可视化


In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13.6: 街景校准结果可视化
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_illusion_comparison(zones_df):
    """绘制统计可达性 vs 真实可达性对比图"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 左图：SAI vs GTA 散点图
    ax1 = axes[0]
    colors = {"severe": "red", "moderate": "orange", 
              "accurate": "green", "underestimated": "blue"}
    
    for level, color in colors.items():
        mask = zones_df["illusion_level"] == level
        if mask.any():
            ax1.scatter(zones_df.loc[mask, "sai"], 
                       zones_df.loc[mask, "gta"],
                       c=color, label=level, alpha=0.7, s=60)
    
    ax1.plot([0, 1], [0, 1], "k--", alpha=0.5, label="完美校准线")
    ax1.set_xlabel("统计可达性 (SAI)", fontsize=12)
    ax1.set_ylabel("真实可达性 (GTA)", fontsize=12)
    ax1.set_title("统计可达性 vs 真实可达性", fontsize=14)
    ax1.legend(loc="upper left")
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    
    # 右图：幻觉分布直方图
    ax2 = axes[1]
    illusion_delta = zones_df["sai"] - zones_df["gta"]
    
    bins = np.linspace(-0.5, 0.5, 21)
    n, bins, patches = ax2.hist(illusion_delta, bins=bins, 
                                 edgecolor="white", alpha=0.7)
    
    for i, patch in enumerate(patches):
        if bins[i] > 0.2:
            patch.set_facecolor("red")
        elif bins[i] > 0.05:
            patch.set_facecolor("orange")
        elif bins[i] < -0.05:
            patch.set_facecolor("blue")
        else:
            patch.set_facecolor("green")
    
    ax2.axvline(x=0, color="black", linestyle="--", alpha=0.5)
    ax2.set_xlabel("幻觉程度 (SAI - GTA)", fontsize=12)
    ax2.set_ylabel("区域数量", fontsize=12)
    ax2.set_title("可达性幻觉分布", fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    legend_elements = [
        mpatches.Patch(color="red", label="严重高估 (>0.2)"),
        mpatches.Patch(color="orange", label="中度高估 (0.05-0.2)"),
        mpatches.Patch(color="green", label="基本准确"),
        mpatches.Patch(color="blue", label="低估 (<-0.05)"),
    ]
    ax2.legend(handles=legend_elements, loc="upper right")
    
    plt.tight_layout()
    return fig


print("=" * 60)
print("[可视化] 校准结果可视化模块已加载")
print("=" * 60)


### 13.7 政策整合与干预建议


In [ ]:
# -*- coding: utf-8 -*-
"""
Section 13.7: 政策整合 — 带温度的空间干预优先级
"""

import numpy as np
import pandas as pd

def generate_intervention_priority(zones_df):
    """生成空间干预优先级列表"""
    if zones_df is None or len(zones_df) == 0:
        print("[政策] 无区域数据，返回空列表")
        return pd.DataFrame()
    
    zones_df = zones_df.copy()
    zones_df["illusion_severity"] = zones_df["illusion_delta"].apply(
        lambda x: 3 if x > 0.3 else 2 if x > 0.15 else 1 if x > 0 else 0
    )
    
    if "avg_perception" in zones_df.columns:
        zones_df["urgency_factor"] = (
            zones_df["illusion_severity"] * 
            (10 - zones_df["avg_perception"].fillna(5))
        )
    else:
        zones_df["urgency_factor"] = zones_df["illusion_severity"] * 5
    
    def suggest_intervention(row):
        illusion = row.get("illusion_delta", 0)
        perception = row.get("avg_perception", 5)
        
        if illusion > 0.3 and perception < 4:
            return "优先: 紧急步行设施改造"
        elif illusion > 0.2:
            return "重要: 增设夜间照明设施"
        elif illusion > 0.1:
            return "一般: 改善人行道连续性"
        elif illusion < -0.1:
            return "关注: 验证统计模型准确性"
        else:
            return "维持现状"
    
    zones_df["intervention_type"] = zones_df.apply(suggest_intervention, axis=1)
    zones_df = zones_df.sort_values("urgency_factor", ascending=False)
    
    return zones_df


print("=" * 60)
print("[政策整合] 政策模块已加载")
print("=" * 60)
